In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ==================================================================================================
# 11_aggregation_reference_crossed_sensitivity_v2.ipynb
# COMPLETE SINGLE-CELL NOTEBOOK
# ==================================================================================================
#
# CORRECTION FROM VERSION 1
# -------------------------
# Version 1 implemented:
#   median_low  = lower middle observed image label
#   median_high = upper middle observed image label
#
# Version 2 implements the frozen Notebook 03 definitions:
#   median_low  = floor of the numerical median image-level MES
#   median_high = ceiling of the numerical median image-level MES
#
# Example:
#   middle labels [0, 2]
#   numerical median = 1
#   v2 median_low = 1
#   v2 median_high = 1
#
# This correction changes only results involving median_low or median_high.
# It does NOT change:
#   - modal_low
#   - modal_high
#   - maximum_label
#   - any image probability
#   - any aggregation prediction
#   - the locked mean-probability/modal-low primary result
#   - the selected Swin or ViT systems
#
# INPUT PROVENANCE
# ----------------
# The notebook uses the exact six development OOF and two locked
# official-partition probability files recorded in the successful
# Notebook 11 v1 source manifest. File hashes are verified before use.
#
# PRIMARY ANALYSIS
# ----------------
# Development OOF probabilities:
#   - selected Swin and ViT systems
#   - seeds 42, 2025, and 2026
#   - five held-out folds pooled within each seed
#   - mean ± sample SD across seeds
#
# DESCRIPTIVE ANALYSIS
# --------------------
# Locked official-partition probabilities:
#   - selected Swin and ViT systems
#   - post-lock descriptive sensitivity only
#   - 5,000 reference-specific stratified patient-bootstrap replicates
#
# OUTPUTS
# -------
# analysis_plan_posthoc_crossed_sensitivity_v2.json
# V1_TO_V2_CORRECTION_RECORD.json
# SUPERSEDES_VERSION_1.txt
# NO_RESELECTION_GUARDRAIL.json
#
# development_crossed_metrics_by_seed.csv
# development_crossed_metrics_summary.csv
# development_reference_distributions.csv
# development_prediction_distributions_by_seed.csv
# development_confusion_matrices_long.csv
#
# official_crossed_metrics_point_estimates.csv
# official_crossed_metrics_with_bootstrap_ci.csv
# official_reference_distributions.csv
# official_prediction_distributions.csv
# official_confusion_matrices_long.csv
#
# v1_to_v2_development_metric_audit.csv
# v1_to_v2_official_metric_audit.csv
# v1_to_v2_reference_distribution_audit.csv
# v1_to_v2_prediction_distribution_audit.csv
#
# Tables_S13_to_S17_v2.xlsx
# Table_S13.md
# Table_S14.md
# Table_S15.md
# Table_S16.md
# Table_S17.md
# Tables_S13_to_S17_v2_combined.txt
#
# Publication figures in PNG, SVG, and PDF
#
# SAFETY
# ------
# This notebook performs:
#   - NO model training
#   - NO checkpoint loading
#   - NO raw-image access
#   - NO model inference
#   - NO calibration fitting
#   - NO threshold optimization
#   - NO model, aggregation, or reference reselection
#
# ==================================================================================================


# ==================================================================================================
# 0. Install missing lightweight dependencies
# ==================================================================================================

import sys
import subprocess
import importlib.util


REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "tabulate": "tabulate",
}


for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )


# ==================================================================================================
# 1. Imports
# ==================================================================================================

import os
import re
import json
import math
import hashlib
import platform
import warnings
import shutil

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

from sklearn.metrics import confusion_matrix
from IPython.display import display


warnings.filterwarnings("ignore")


# ==================================================================================================
# 2. Mount Google Drive
# ==================================================================================================

try:
    from google.colab import drive

    (drive.mount(
        "/content/drive",
        force_remount=False,
    ) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)

except Exception as mount_error:
    print("Google Drive was not mounted automatically.")
    print("Reason:", repr(mount_error))


# ==================================================================================================
# 3. Fixed settings
# ==================================================================================================

NUM_CLASSES = 4

CLASS_IDS = [0, 1, 2, 3]

CLASS_NAMES = {
    0: "Mayo 0",
    1: "Mayo 1",
    2: "Mayo 2",
    3: "Mayo 3",
}

CLASS_VALUES = np.asarray(
    CLASS_IDS,
    dtype=float,
)

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


TRAINING_SEEDS = [
    42,
    2025,
    2026,
]


EXPECTED_DEVELOPMENT_IMAGES = 9590
EXPECTED_DEVELOPMENT_PATIENTS = 478

EXPECTED_OFFICIAL_IMAGES = 1686
EXPECTED_OFFICIAL_PATIENTS = 86


EXPECTED_DEVELOPMENT_PRIMARY_QWK = {
    ("Swin", 42): 0.866978,
    ("Swin", 2025): 0.877706,
    ("Swin", 2026): 0.885770,
    ("ViT", 42): 0.842837,
    ("ViT", 2025): 0.839909,
    ("ViT", 2026): 0.865410,
}


EXPECTED_OFFICIAL_PRIMARY_QWK = {
    "Swin": 0.7846874495235019,
    "ViT": 0.7505,
}


SOURCE_QWK_TOLERANCE = 0.010


# Frozen Notebook 03 development reference distributions.
EXPECTED_DEVELOPMENT_REFERENCE_COUNTS = {
    "modal_low": {
        0: 293,
        1: 108,
        2: 46,
        3: 31,
    },
    "modal_high": {
        0: 275,
        1: 112,
        2: 55,
        3: 36,
    },
    "median_low": {
        0: 259,
        1: 151,
        2: 49,
        3: 19,
    },
    "median_high": {
        0: 250,
        1: 153,
        2: 53,
        3: 22,
    },
    "maximum_label": {
        0: 64,
        1: 143,
        2: 117,
        3: 154,
    },
}


BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20260831

TRIM_FRACTION = 0.10
MIN_IMAGES_FOR_CONFIDENCE_FILTER = 5

PNG_DPI = 600

SAVE_PNG = True
SAVE_SVG = True
SAVE_PDF = True

SHOW_FIGURES = True


PRIMARY_AGGREGATION = "mean_probability_argmax"
PRIMARY_REFERENCE = "modal_low"


# ==================================================================================================
# 4. Resolve directories
# ==================================================================================================

REVISION_ROOT_CANDIDATES = [
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
]


REVISION_ROOT = next(
    (
        path
        for path in REVISION_ROOT_CANDIDATES
        if path.exists()
    ),
    None,
)


if REVISION_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the revision_2026 directory.\n"
        "Checked:\n"
        + "\n".join(
            str(path)
            for path in REVISION_ROOT_CANDIDATES
        )
    )


V1_ROOT = (
    REVISION_ROOT
    / "11_aggregation_reference_crossed_sensitivity_v1"
)

V2_ROOT = (
    REVISION_ROOT
    / "11_aggregation_reference_crossed_sensitivity_v2"
)


V1_SOURCE_MANIFEST = (
    V1_ROOT
    / "manifests"
    / "crossed_sensitivity_source_manifest.csv"
)


if not V1_ROOT.exists():
    raise FileNotFoundError(
        "Notebook 11 version 1 output directory was not found:\n"
        + str(V1_ROOT)
    )


if not V1_SOURCE_MANIFEST.exists():
    raise FileNotFoundError(
        "The Notebook 11 v1 source manifest was not found:\n"
        + str(V1_SOURCE_MANIFEST)
        + "\n\nVersion 2 intentionally reuses and verifies the exact "
          "input files selected by the successful v1 analysis."
    )


SOURCE_DATA_DIR = (
    V2_ROOT
    / "source_data"
)

MANIFEST_DIR = (
    V2_ROOT
    / "manifests"
)

FIGURE_ROOT = (
    V2_ROOT
    / "figures"
)

PNG_DIR = (
    FIGURE_ROOT
    / "png"
)

SVG_DIR = (
    FIGURE_ROOT
    / "svg"
)

PDF_DIR = (
    FIGURE_ROOT
    / "pdf"
)

PUBLICATION_TABLE_DIR = (
    V2_ROOT
    / "publication_tables_S13_S17"
)


for directory in [
    V2_ROOT,
    SOURCE_DATA_DIR,
    MANIFEST_DIR,
    FIGURE_ROOT,
    PNG_DIR,
    SVG_DIR,
    PDF_DIR,
    PUBLICATION_TABLE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("=" * 125)
print("11_aggregation_reference_crossed_sensitivity_v2")
print("=" * 125)

print("Revision root:", REVISION_ROOT)
print("Version 1 root:", V1_ROOT)
print("Version 1 source manifest:", V1_SOURCE_MANIFEST)
print("Version 2 output root:", V2_ROOT)


# ==================================================================================================
# 5. Safety status
# ==================================================================================================

TRAINING_PERFORMED = False
CHECKPOINTS_LOADED = False
RAW_IMAGES_ACCESSED = False
MODEL_INFERENCE_PERFORMED = False
CALIBRATION_FITTED = False
THRESHOLD_OPTIMIZED = False
MODEL_SELECTION_PERFORMED = False
AGGREGATION_RESELECTION_PERFORMED = False
REFERENCE_RESELECTION_PERFORMED = False
PRIMARY_ANALYSIS_CHANGED = False


print("")
print("Safety status:")
print("  Training performed:", TRAINING_PERFORMED)
print("  Checkpoints loaded:", CHECKPOINTS_LOADED)
print("  Raw images accessed:", RAW_IMAGES_ACCESSED)
print("  Model inference performed:", MODEL_INFERENCE_PERFORMED)
print("  Calibration fitted:", CALIBRATION_FITTED)
print("  Threshold optimized:", THRESHOLD_OPTIMIZED)
print("  Model selection performed:", MODEL_SELECTION_PERFORMED)
print(
    "  Aggregation reselection performed:",
    AGGREGATION_RESELECTION_PERFORMED,
)
print(
    "  Reference reselection performed:",
    REFERENCE_RESELECTION_PERFORMED,
)
print("  Primary analysis changed:", PRIMARY_ANALYSIS_CHANGED)


# ==================================================================================================
# 6. Analysis definitions
# ==================================================================================================

AGGREGATION_RULES = [
    {
        "key": "mean_probability_argmax",
        "display_name": "Mean probability",
        "origin": "development_prespecified",
        "is_primary": True,
    },
    {
        "key": "median_probability_argmax",
        "display_name": "Median probability",
        "origin": "development_prespecified",
        "is_primary": False,
    },
    {
        "key": "majority_vote_low_tie",
        "display_name": "Majority vote (lower-class tie)",
        "origin": "development_prespecified",
        "is_primary": False,
    },
    {
        "key": "mean_expected_mes_round",
        "display_name": "Mean expected MES",
        "origin": "development_prespecified",
        "is_primary": False,
    },
    {
        "key": "median_expected_mes_round",
        "display_name": "Median expected MES",
        "origin": "development_prespecified",
        "is_primary": False,
    },
    {
        "key": "confidence_weighted_mean_argmax",
        "display_name": "Confidence-weighted mean",
        "origin": "post_lock_exploratory",
        "is_primary": False,
    },
    {
        "key": "confidence_quantile_filtered_mean_argmax",
        "display_name": "Confidence-quantile-filtered mean",
        "origin": "post_lock_exploratory",
        "is_primary": False,
    },
    {
        "key": "maximum_predicted_image_class",
        "display_name": "Maximum predicted image class",
        "origin": "development_prespecified",
        "is_primary": False,
    },
]


REFERENCE_RULES = [
    {
        "key": "modal_low",
        "display_name": "Modal-low",
        "definition": (
            "Modal image-level MES with lower-class tie resolution"
        ),
        "is_primary": True,
    },
    {
        "key": "modal_high",
        "display_name": "Modal-high",
        "definition": (
            "Modal image-level MES with higher-class tie resolution"
        ),
        "is_primary": False,
    },
    {
        "key": "median_low",
        "display_name": "Median-low",
        "definition": (
            "Floor of the numerical median image-level MES"
        ),
        "is_primary": False,
    },
    {
        "key": "median_high",
        "display_name": "Median-high",
        "definition": (
            "Ceiling of the numerical median image-level MES"
        ),
        "is_primary": False,
    },
    {
        "key": "maximum_label",
        "display_name": "Maximum label",
        "definition": (
            "Maximum observed image-level MES"
        ),
        "is_primary": False,
    },
]


METRIC_NAMES = [
    "qwk",
    "mae",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
]


AGGREGATION_ORDER = [
    item["key"]
    for item in AGGREGATION_RULES
]

REFERENCE_ORDER = [
    item["key"]
    for item in REFERENCE_RULES
]

AGGREGATION_LABELS = {
    item["key"]: item["display_name"]
    for item in AGGREGATION_RULES
}

REFERENCE_LABELS = {
    item["key"]: item["display_name"]
    for item in REFERENCE_RULES
}


# ==================================================================================================
# 7. Atomic output and hash helpers
# ==================================================================================================

def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_json_save(
    payload,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_text_save(
    text,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(text)

        file.flush()
        os.fsync(
            file.fileno()
        )

    os.replace(
        temporary_path,
        path,
    )


def file_sha256(
    path,
    block_size=8 * 1024 * 1024,
):
    path = Path(path)

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def json_fingerprint(
    payload,
):
    canonical = json.dumps(
        payload,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        canonical
    ).hexdigest()


for abandoned_file in V2_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ==================================================================================================
# 8. Freeze the corrected version 2 plan
# ==================================================================================================

ANALYSIS_SPECIFICATION = {
    "protocol_version": (
        "posthoc_aggregation_reference_crossed_sensitivity_v2"
    ),
    "supersedes": (
        "posthoc_aggregation_reference_crossed_sensitivity_v1"
    ),
    "correction_reason": (
        "Version 1 implemented median-low and median-high as "
        "lower and upper middle observations. Version 2 uses "
        "the frozen Notebook 03 definitions: floor and ceiling "
        "of the numerical median image-level MES."
    ),
    "analysis_role": (
        "post hoc response-to-reviewer estimand-alignment sensitivity"
    ),
    "primary_data_source": (
        "development pooled OOF image probabilities"
    ),
    "official_partition_role": (
        "post-lock descriptive sensitivity only"
    ),
    "systems": [
        "Swin",
        "ViT",
    ],
    "development_seeds": TRAINING_SEEDS,
    "aggregation_rules": AGGREGATION_RULES,
    "reference_rules": REFERENCE_RULES,
    "metrics": METRIC_NAMES,
    "rounding_rule": (
        "Half-up rounding to nearest integer class, clipped to 0–3"
    ),
    "confidence_weight": (
        "Maximum softmax probability of each image"
    ),
    "confidence_quantile_filter": {
        "lower_quantile": TRIM_FRACTION,
        "upper_quantile": (
            1.0 - TRIM_FRACTION
        ),
        "minimum_images": (
            MIN_IMAGES_FOR_CONFIDENCE_FILTER
        ),
        "boundary_inclusion": "inclusive",
    },
    "official_bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "stratification": (
            "Within the derived reference used in each matrix column"
        ),
        "interval": (
            "2.5th and 97.5th percentiles"
        ),
        "compatibility": (
            "Same system-specific and reference-specific RNG "
            "construction as version 1, so nonmedian intervals "
            "remain directly auditable."
        ),
    },
    "expected_development_reference_counts": (
        EXPECTED_DEVELOPMENT_REFERENCE_COUNTS
    ),
    "guardrails": [
        "No model training",
        "No checkpoint loading",
        "No raw-image access",
        "No model inference",
        "No calibration fitting",
        "No threshold optimization",
        "No aggregation-rule reselection",
        "No patient-reference reselection",
        "No change to the primary Swin system",
        "No change to the prespecified ViT comparator",
        (
            "No replacement of the locked mean-probability/"
            "modal-low primary analysis"
        ),
    ],
}


ANALYSIS_PLAN_FINGERPRINT = json_fingerprint(
    ANALYSIS_SPECIFICATION
)


ANALYSIS_PLAN_PATH = (
    V2_ROOT
    / "analysis_plan_posthoc_crossed_sensitivity_v2.json"
)


if ANALYSIS_PLAN_PATH.exists():
    with open(
        ANALYSIS_PLAN_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(
            file
        )

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        !=
        ANALYSIS_PLAN_FINGERPRINT
    ):
        raise RuntimeError(
            "An existing version 2 output directory contains a "
            "different frozen analysis plan.\n\n"
            "Do not overwrite it. Archive the existing directory "
            "or use a new versioned output directory."
        )

    print("")
    print("Existing version 2 plan validated:")
    print(ANALYSIS_PLAN_PATH)

else:
    plan_payload = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "analysis_plan_fingerprint": (
            ANALYSIS_PLAN_FINGERPRINT
        ),
        "analysis_specification": (
            ANALYSIS_SPECIFICATION
        ),
    }

    atomic_json_save(
        plan_payload,
        ANALYSIS_PLAN_PATH,
    )

    print("")
    print("New version 2 plan written:")
    print(ANALYSIS_PLAN_PATH)


print("Version 2 analysis-plan fingerprint:")
print(ANALYSIS_PLAN_FINGERPRINT)


# ==================================================================================================
# 9. Dataframe helpers
# ==================================================================================================

def normalize_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def find_column(
    dataframe,
    aliases,
):
    normalized_to_original = {
        normalize_name(column): column
        for column in dataframe.columns
    }

    for alias in aliases:
        normalized_alias = normalize_name(
            alias
        )

        if normalized_alias in normalized_to_original:
            return normalized_to_original[
                normalized_alias
            ]

    return None


def identify_probability_columns(
    dataframe,
):
    normalized_to_original = {
        normalize_name(column): column
        for column in dataframe.columns
    }

    selected_columns = []

    for class_id in CLASS_IDS:
        aliases = [
            f"prob_{class_id}",
            f"probability_{class_id}",
            f"p_{class_id}",
            f"class_{class_id}_prob",
            f"class_{class_id}_probability",
            f"mayo_{class_id}_prob",
            f"mayo_{class_id}_probability",
        ]

        selected = None

        for alias in aliases:
            normalized_alias = normalize_name(
                alias
            )

            if normalized_alias in normalized_to_original:
                selected = normalized_to_original[
                    normalized_alias
                ]
                break

        if selected is None:
            suffix_patterns = [
                re.compile(
                    rf".*prob_?{class_id}$"
                ),
                re.compile(
                    rf".*probability_?{class_id}$"
                ),
            ]

            candidates = []

            for normalized_column, original_column in (
                normalized_to_original.items()
            ):
                if any(
                    pattern.fullmatch(
                        normalized_column
                    )
                    for pattern in suffix_patterns
                ):
                    candidates.append(
                        original_column
                    )

            if len(candidates) == 1:
                selected = candidates[0]

        if selected is None:
            return []

        selected_columns.append(
            selected
        )

    return selected_columns


def infer_system(
    value,
):
    text = str(value).lower()

    if "swin" in text:
        return "Swin"

    if (
        re.search(
            r"(^|[/_\-])vit([/_\-]|$)",
            text,
        )
        is not None
        or
        "vision_transformer"
        in text
        or
        "vit_l"
        in text
    ):
        return "ViT"

    return None


def normalize_patient_ids(
    series,
):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
    )


def standardize_prediction_source(
    path,
    expected_system=None,
    expected_seed=None,
):
    raw_dataframe = pd.read_csv(
        path,
        low_memory=False,
    )

    system_column = find_column(
        raw_dataframe,
        aliases=[
            "system",
            "system_name",
            "backbone",
            "backbone_name",
            "model",
            "model_name",
            "candidate",
            "candidate_name",
            "representative_configuration",
            "configuration",
        ],
    )

    seed_column = find_column(
        raw_dataframe,
        aliases=[
            "training_seed",
            "seed",
            "random_seed",
        ],
    )

    dataframe = raw_dataframe.copy()

    if (
        expected_system is not None
        and
        system_column is not None
    ):
        parsed_system = dataframe[
            system_column
        ].map(
            infer_system
        )

        matching_mask = (
            parsed_system
            ==
            expected_system
        )

        if matching_mask.any():
            dataframe = dataframe.loc[
                matching_mask
            ].copy()

    if (
        expected_seed is not None
        and
        seed_column is not None
    ):
        numeric_seed = pd.to_numeric(
            dataframe[
                seed_column
            ],
            errors="coerce",
        )

        matching_mask = (
            numeric_seed
            ==
            int(
                expected_seed
            )
        )

        if matching_mask.any():
            dataframe = dataframe.loc[
                matching_mask
            ].copy()

    patient_column = find_column(
        dataframe,
        aliases=[
            "patient_id",
            "patient",
            "pid",
            "group_id",
        ],
    )

    true_column = find_column(
        dataframe,
        aliases=[
            "y_true",
            "true_label",
            "image_label",
            "label",
            "target",
            "mes_label",
            "mayo_label",
            "mayo_class",
        ],
    )

    path_column = find_column(
        dataframe,
        aliases=[
            "path",
            "image_path",
            "filepath",
            "file_path",
            "filename",
            "image_id",
        ],
    )

    probability_source_columns = (
        identify_probability_columns(
            dataframe
        )
    )

    if patient_column is None:
        raise ValueError(
            "Could not identify a patient-ID column in:\n"
            + str(path)
        )

    if true_column is None:
        raise ValueError(
            "Could not identify a true-label column in:\n"
            + str(path)
        )

    if (
        len(
            probability_source_columns
        )
        !=
        NUM_CLASSES
    ):
        raise ValueError(
            "Could not identify exactly four probability columns in:\n"
            + str(path)
        )

    output = pd.DataFrame()

    output["patient_id"] = normalize_patient_ids(
        dataframe[
            patient_column
        ]
    )

    output["y_true"] = pd.to_numeric(
        dataframe[
            true_column
        ],
        errors="coerce",
    )

    if path_column is not None:
        output["path"] = dataframe[
            path_column
        ].astype(str)

    else:
        output["path"] = [
            f"row_{row_index:07d}"
            for row_index in range(
                len(
                    dataframe
                )
            )
        ]

    for class_id, source_column in enumerate(
        probability_source_columns
    ):
        output[
            f"prob_{class_id}"
        ] = pd.to_numeric(
            dataframe[
                source_column
            ],
            errors="coerce",
        )

    output = output.dropna(
        subset=[
            "patient_id",
            "y_true",
            "prob_0",
            "prob_1",
            "prob_2",
            "prob_3",
        ]
    ).copy()

    output["y_true"] = output[
        "y_true"
    ].astype(int)

    invalid_labels = set(
        output[
            "y_true"
        ].unique()
    ) - set(
        CLASS_IDS
    )

    if invalid_labels:
        raise ValueError(
            "Invalid MES labels found in:\n"
            + str(path)
            + "\n"
            + repr(
                invalid_labels
            )
        )

    probabilities = output[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=float
    )

    probabilities = np.clip(
        probabilities,
        0.0,
        None,
    )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(
        row_sums <= 0
    ):
        raise ValueError(
            "At least one probability row has a nonpositive sum in:\n"
            + str(path)
        )

    probabilities = (
        probabilities
        /
        row_sums
    )

    output[
        PROBABILITY_COLUMNS
    ] = probabilities

    output["y_pred"] = np.argmax(
        probabilities,
        axis=1,
    ).astype(int)

    return output.reset_index(
        drop=True
    )


# ==================================================================================================
# 10. Corrected reference functions
# ==================================================================================================

def modal_low(
    values,
):
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts
            ==
            counts.max()
        )[0]
    )


def modal_high(
    values,
):
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts
            ==
            counts.max()
        )[-1]
    )


def median_low(
    values,
):
    """
    Frozen Notebook 03 definition:
    floor of the numerical median image-level MES.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    return int(
        np.floor(
            np.median(
                values
            )
        )
    )


def median_high(
    values,
):
    """
    Frozen Notebook 03 definition:
    ceiling of the numerical median image-level MES.
    """
    values = np.asarray(
        values,
        dtype=float,
    )

    return int(
        np.ceil(
            np.median(
                values
            )
        )
    )


def maximum_label(
    values,
):
    return int(
        np.max(
            np.asarray(
                values,
                dtype=int,
            )
        )
    )


# ==================================================================================================
# 11. Aggregation functions
# ==================================================================================================

def normalize_probability_vector(
    vector,
):
    vector = np.asarray(
        vector,
        dtype=float,
    )

    vector = np.clip(
        vector,
        0.0,
        None,
    )

    total = float(
        vector.sum()
    )

    if total <= 0:
        return np.ones(
            NUM_CLASSES,
            dtype=float,
        ) / NUM_CLASSES

    return (
        vector
        /
        total
    )


def round_half_up_to_class(
    value,
):
    rounded = int(
        math.floor(
            float(value)
            +
            0.5
        )
    )

    return int(
        np.clip(
            rounded,
            0,
            NUM_CLASSES - 1,
        )
    )


def majority_vote_low_tie(
    hard_predictions,
):
    counts = np.bincount(
        np.asarray(
            hard_predictions,
            dtype=int,
        ),
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts
            ==
            counts.max()
        )[0]
    )


def aggregate_patient_prediction(
    probabilities,
    rule_key,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    hard_predictions = np.argmax(
        probabilities,
        axis=1,
    )

    mean_probabilities = (
        normalize_probability_vector(
            probabilities.mean(
                axis=0
            )
        )
    )

    if rule_key == "mean_probability_argmax":
        return int(
            np.argmax(
                mean_probabilities
            )
        )

    if rule_key == "median_probability_argmax":
        median_probabilities = (
            normalize_probability_vector(
                np.median(
                    probabilities,
                    axis=0,
                )
            )
        )

        return int(
            np.argmax(
                median_probabilities
            )
        )

    if rule_key == "majority_vote_low_tie":
        return majority_vote_low_tie(
            hard_predictions
        )

    expected_mes = (
        probabilities
        @
        CLASS_VALUES
    )

    if rule_key == "mean_expected_mes_round":
        return round_half_up_to_class(
            expected_mes.mean()
        )

    if rule_key == "median_expected_mes_round":
        return round_half_up_to_class(
            np.median(
                expected_mes
            )
        )

    if rule_key == "confidence_weighted_mean_argmax":
        confidence = probabilities.max(
            axis=1
        )

        if float(
            confidence.sum()
        ) <= 0:
            weighted_probabilities = (
                mean_probabilities
            )

        else:
            weighted_probabilities = (
                normalize_probability_vector(
                    (
                        probabilities
                        *
                        confidence[
                            :,
                            None,
                        ]
                    ).sum(
                        axis=0
                    )
                )
            )

        return int(
            np.argmax(
                weighted_probabilities
            )
        )

    if (
        rule_key
        ==
        "confidence_quantile_filtered_mean_argmax"
    ):
        if (
            len(
                probabilities
            )
            <
            MIN_IMAGES_FOR_CONFIDENCE_FILTER
        ):
            filtered_probabilities = (
                mean_probabilities
            )

        else:
            confidence = probabilities.max(
                axis=1
            )

            lower_boundary = np.quantile(
                confidence,
                TRIM_FRACTION,
            )

            upper_boundary = np.quantile(
                confidence,
                1.0 - TRIM_FRACTION,
            )

            keep_mask = (
                (
                    confidence
                    >=
                    lower_boundary
                )
                &
                (
                    confidence
                    <=
                    upper_boundary
                )
            )

            retained = probabilities[
                keep_mask
            ]

            if len(
                retained
            ) == 0:
                retained = probabilities

            filtered_probabilities = (
                normalize_probability_vector(
                    retained.mean(
                        axis=0
                    )
                )
            )

        return int(
            np.argmax(
                filtered_probabilities
            )
        )

    if rule_key == "maximum_predicted_image_class":
        return int(
            hard_predictions.max()
        )

    raise ValueError(
        "Unknown aggregation rule: "
        + str(
            rule_key
        )
    )


def build_patient_table(
    image_dataframe,
):
    rows = []

    for patient_id, group in image_dataframe.groupby(
        "patient_id",
        sort=True,
    ):
        labels = group[
            "y_true"
        ].to_numpy(
            dtype=int
        )

        probabilities = group[
            PROBABILITY_COLUMNS
        ].to_numpy(
            dtype=float
        )

        row = {
            "patient_id": str(
                patient_id
            ),
            "n_images": int(
                len(
                    group
                )
            ),
            "modal_low": modal_low(
                labels
            ),
            "modal_high": modal_high(
                labels
            ),
            "median_low": median_low(
                labels
            ),
            "median_high": median_high(
                labels
            ),
            "maximum_label": maximum_label(
                labels
            ),
        }

        for aggregation_rule in AGGREGATION_RULES:
            rule_key = aggregation_rule[
                "key"
            ]

            row[
                rule_key
            ] = aggregate_patient_prediction(
                probabilities=probabilities,
                rule_key=rule_key,
            )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    ).sort_values(
        "patient_id"
    ).reset_index(
        drop=True
    )


# ==================================================================================================
# 12. Metric implementation
# ==================================================================================================

def metrics_from_confusion_matrix(
    matrix,
):
    matrix = np.asarray(
        matrix,
        dtype=float,
    )

    total = float(
        matrix.sum()
    )

    if total <= 0:
        return {
            metric_name: np.nan
            for metric_name in METRIC_NAMES
        }

    row_sums = matrix.sum(
        axis=1
    )

    column_sums = matrix.sum(
        axis=0
    )

    quadratic_weights = np.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=float,
    )

    absolute_distances = np.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=float,
    )

    for true_class in CLASS_IDS:
        for predicted_class in CLASS_IDS:
            quadratic_weights[
                true_class,
                predicted_class,
            ] = (
                (
                    true_class
                    -
                    predicted_class
                )
                ** 2
                /
                (
                    (
                        NUM_CLASSES
                        -
                        1
                    )
                    ** 2
                )
            )

            absolute_distances[
                true_class,
                predicted_class,
            ] = abs(
                true_class
                -
                predicted_class
            )

    expected_counts = np.outer(
        row_sums,
        column_sums,
    ) / total

    observed_weighted_disagreement = float(
        np.sum(
            quadratic_weights
            *
            matrix
        )
    )

    expected_weighted_disagreement = float(
        np.sum(
            quadratic_weights
            *
            expected_counts
        )
    )

    if expected_weighted_disagreement <= 0:
        qwk = np.nan

    else:
        qwk = (
            1.0
            -
            observed_weighted_disagreement
            /
            expected_weighted_disagreement
        )

    mae = float(
        np.sum(
            absolute_distances
            *
            matrix
        )
        /
        total
    )

    accuracy = float(
        np.trace(
            matrix
        )
        /
        total
    )

    recalls = np.divide(
        np.diag(
            matrix
        ),
        row_sums,
        out=np.zeros(
            NUM_CLASSES,
            dtype=float,
        ),
        where=row_sums > 0,
    )

    reference_classes_present = (
        row_sums > 0
    )

    balanced_accuracy = float(
        recalls[
            reference_classes_present
        ].mean()
    )

    precisions = np.divide(
        np.diag(
            matrix
        ),
        column_sums,
        out=np.zeros(
            NUM_CLASSES,
            dtype=float,
        ),
        where=column_sums > 0,
    )

    f1_values = np.divide(
        2.0
        *
        precisions
        *
        recalls,
        precisions
        +
        recalls,
        out=np.zeros(
            NUM_CLASSES,
            dtype=float,
        ),
        where=(
            precisions
            +
            recalls
        )
        >
        0,
    )

    macro_f1 = float(
        f1_values.mean()
    )

    return {
        "qwk": float(
            qwk
        ),
        "mae": mae,
        "accuracy": accuracy,
        "balanced_accuracy": (
            balanced_accuracy
        ),
        "macro_f1": macro_f1,
    }


def compute_metrics(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        np.asarray(
            y_true,
            dtype=int,
        ),
        np.asarray(
            y_pred,
            dtype=int,
        ),
        labels=CLASS_IDS,
    )

    return metrics_from_confusion_matrix(
        matrix
    )


def append_confusion_rows(
    output_rows,
    split,
    system,
    training_seed,
    aggregation_rule,
    reference_rule,
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        np.asarray(
            y_true,
            dtype=int,
        ),
        np.asarray(
            y_pred,
            dtype=int,
        ),
        labels=CLASS_IDS,
    )

    for true_class in CLASS_IDS:
        for predicted_class in CLASS_IDS:
            output_rows.append(
                {
                    "split": split,
                    "system": system,
                    "training_seed": (
                        training_seed
                    ),
                    "aggregation_rule": (
                        aggregation_rule
                    ),
                    "reference_rule": (
                        reference_rule
                    ),
                    "true_class": int(
                        true_class
                    ),
                    "predicted_class": int(
                        predicted_class
                    ),
                    "count": int(
                        matrix[
                            true_class,
                            predicted_class,
                        ]
                    ),
                }
            )


# ==================================================================================================
# 13. Load and verify the exact version 1 source manifest
# ==================================================================================================

v1_source_manifest_df = pd.read_csv(
    V1_SOURCE_MANIFEST,
    low_memory=False,
)


required_manifest_columns = {
    "split",
    "system",
    "training_seed",
    "path",
    "sha256",
}


missing_manifest_columns = (
    required_manifest_columns
    -
    set(
        v1_source_manifest_df.columns
    )
)


if missing_manifest_columns:
    raise ValueError(
        "The v1 source manifest is missing required columns:\n"
        + repr(
            sorted(
                missing_manifest_columns
            )
        )
    )


selected_sources = {}


for _, manifest_row in v1_source_manifest_df.iterrows():
    split = str(
        manifest_row[
            "split"
        ]
    ).strip()

    system = str(
        manifest_row[
            "system"
        ]
    ).strip()

    source_path = Path(
        str(
            manifest_row[
                "path"
            ]
        )
    )

    if not source_path.exists():
        raise FileNotFoundError(
            "A source recorded in the v1 manifest no longer exists:\n"
            + str(
                source_path
            )
        )

    observed_hash = file_sha256(
        source_path
    )

    expected_hash = str(
        manifest_row[
            "sha256"
        ]
    ).strip()

    if observed_hash != expected_hash:
        raise RuntimeError(
            "A source file hash differs from the successful "
            "version 1 manifest.\n\n"
            "Path:\n"
            + str(
                source_path
            )
            + "\n\nExpected SHA-256:\n"
            + expected_hash
            + "\n\nObserved SHA-256:\n"
            + observed_hash
        )

    if split == "development_oof":
        training_seed = int(
            float(
                manifest_row[
                    "training_seed"
                ]
            )
        )

        source_dataframe = (
            standardize_prediction_source(
                source_path,
                expected_system=system,
                expected_seed=training_seed,
            )
        )

        selected_sources[
            (
                split,
                system,
                training_seed,
            )
        ] = {
            "path": source_path,
            "dataframe": source_dataframe,
            "sha256": observed_hash,
        }

    elif split == "official_partition":
        source_dataframe = (
            standardize_prediction_source(
                source_path,
                expected_system=system,
                expected_seed=None,
            )
        )

        selected_sources[
            (
                split,
                system,
                None,
            )
        ] = {
            "path": source_path,
            "dataframe": source_dataframe,
            "sha256": observed_hash,
        }

    else:
        raise ValueError(
            "Unexpected split in v1 source manifest: "
            + split
        )


expected_source_keys = {
    (
        "development_oof",
        system,
        training_seed,
    )
    for system in [
        "Swin",
        "ViT",
    ]
    for training_seed in TRAINING_SEEDS
}


expected_source_keys.update(
    {
        (
            "official_partition",
            "Swin",
            None,
        ),
        (
            "official_partition",
            "ViT",
            None,
        ),
    }
)


if set(
    selected_sources.keys()
) != expected_source_keys:
    raise RuntimeError(
        "The v1 source manifest does not identify exactly "
        "six development OOF and two official-partition sources."
    )


# Preserve the exact source manifest in v2.
shutil.copy2(
    V1_SOURCE_MANIFEST,
    MANIFEST_DIR
    / "crossed_sensitivity_source_manifest.csv",
)


# ==================================================================================================
# 14. Validate source sizes and locked primary values
# ==================================================================================================

source_validation_rows = []


for source_key, source_record in selected_sources.items():
    split, system, training_seed = source_key

    dataframe = source_record[
        "dataframe"
    ]

    patient_table = build_patient_table(
        dataframe
    )

    primary_metrics = compute_metrics(
        patient_table[
            PRIMARY_REFERENCE
        ],
        patient_table[
            PRIMARY_AGGREGATION
        ],
    )

    if split == "development_oof":
        expected_images = (
            EXPECTED_DEVELOPMENT_IMAGES
        )

        expected_patients = (
            EXPECTED_DEVELOPMENT_PATIENTS
        )

        expected_qwk = (
            EXPECTED_DEVELOPMENT_PRIMARY_QWK[
                (
                    system,
                    training_seed,
                )
            ]
        )

    else:
        expected_images = (
            EXPECTED_OFFICIAL_IMAGES
        )

        expected_patients = (
            EXPECTED_OFFICIAL_PATIENTS
        )

        expected_qwk = (
            EXPECTED_OFFICIAL_PRIMARY_QWK[
                system
            ]
        )

    if len(
        dataframe
    ) != expected_images:
        raise RuntimeError(
            f"{split} {system} source has "
            f"{len(dataframe)} images; expected {expected_images}."
        )

    if len(
        patient_table
    ) != expected_patients:
        raise RuntimeError(
            f"{split} {system} source has "
            f"{len(patient_table)} patients; expected {expected_patients}."
        )

    if abs(
        primary_metrics[
            "qwk"
        ]
        -
        expected_qwk
    ) > SOURCE_QWK_TOLERANCE:
        raise RuntimeError(
            "A source does not reproduce the expected locked "
            "mean-probability/modal-low QWK.\n"
            f"Split: {split}\n"
            f"System: {system}\n"
            f"Seed: {training_seed}\n"
            f"Observed: {primary_metrics['qwk']:.6f}\n"
            f"Expected: {expected_qwk:.6f}"
        )

    source_validation_rows.append(
        {
            "split": split,
            "system": system,
            "training_seed": training_seed,
            "path": str(
                source_record[
                    "path"
                ]
            ),
            "sha256": source_record[
                "sha256"
            ],
            "n_images": int(
                len(
                    dataframe
                )
            ),
            "n_patients": int(
                len(
                    patient_table
                )
            ),
            "primary_patient_qwk": float(
                primary_metrics[
                    "qwk"
                ]
            ),
        }
    )


source_validation_df = pd.DataFrame(
    source_validation_rows
).sort_values(
    [
        "split",
        "system",
        "training_seed",
    ],
    na_position="last",
).reset_index(
    drop=True
)


atomic_csv_save(
    source_validation_df,
    MANIFEST_DIR
    / "v2_source_validation.csv",
    index=False,
)


print("")
print("=" * 125)
print("VERSION 2 SOURCE VALIDATION")
print("=" * 125)

display(
    source_validation_df
)


# ==================================================================================================
# 15. Build development and official patient tables
# ==================================================================================================

development_patient_tables = {}


for system in [
    "Swin",
    "ViT",
]:
    for training_seed in TRAINING_SEEDS:
        source_dataframe = selected_sources[
            (
                "development_oof",
                system,
                training_seed,
            )
        ][
            "dataframe"
        ]

        development_patient_tables[
            (
                system,
                training_seed,
            )
        ] = build_patient_table(
            source_dataframe
        )


official_patient_tables = {}


for system in [
    "Swin",
    "ViT",
]:
    source_dataframe = selected_sources[
        (
            "official_partition",
            system,
            None,
        )
    ][
        "dataframe"
    ]

    official_patient_tables[
        system
    ] = build_patient_table(
        source_dataframe
    )


# ==================================================================================================
# 16. Validate reference identity across all sources
# ==================================================================================================

reference_columns = REFERENCE_ORDER


canonical_development_reference_table = (
    development_patient_tables[
        (
            "Swin",
            42,
        )
    ][
        [
            "patient_id",
        ]
        +
        reference_columns
    ]
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


for source_key, patient_table in (
    development_patient_tables.items()
):
    comparison = (
        canonical_development_reference_table.merge(
            patient_table[
                [
                    "patient_id",
                ]
                +
                reference_columns
            ],
            on="patient_id",
            how="inner",
            suffixes=(
                "_canonical",
                "_source",
            ),
            validate="one_to_one",
        )
    )

    if len(
        comparison
    ) != EXPECTED_DEVELOPMENT_PATIENTS:
        raise RuntimeError(
            "Development sources do not contain the same 478 patients."
        )

    for reference_rule in reference_columns:
        disagreement_count = int(
            (
                comparison[
                    reference_rule
                    +
                    "_canonical"
                ]
                !=
                comparison[
                    reference_rule
                    +
                    "_source"
                ]
            ).sum()
        )

        if disagreement_count > 0:
            raise RuntimeError(
                "Development reference labels differ across sources.\n"
                f"Source: {source_key}\n"
                f"Reference: {reference_rule}\n"
                f"Disagreements: {disagreement_count}"
            )


canonical_official_reference_table = (
    official_patient_tables[
        "Swin"
    ][
        [
            "patient_id",
        ]
        +
        reference_columns
    ]
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


official_reference_comparison = (
    canonical_official_reference_table.merge(
        official_patient_tables[
            "ViT"
        ][
            [
                "patient_id",
            ]
            +
            reference_columns
        ],
        on="patient_id",
        how="inner",
        suffixes=(
            "_swin",
            "_vit",
        ),
        validate="one_to_one",
    )
)


if len(
    official_reference_comparison
) != EXPECTED_OFFICIAL_PATIENTS:
    raise RuntimeError(
        "Official Swin and ViT sources do not contain "
        "the same 86 patients."
    )


for reference_rule in reference_columns:
    disagreement_count = int(
        (
            official_reference_comparison[
                reference_rule
                +
                "_swin"
            ]
            !=
            official_reference_comparison[
                reference_rule
                +
                "_vit"
            ]
        ).sum()
    )

    if disagreement_count > 0:
        raise RuntimeError(
            "Official reference labels differ between systems.\n"
            f"Reference: {reference_rule}\n"
            f"Disagreements: {disagreement_count}"
        )


# ==================================================================================================
# 17. Validate corrected development reference counts
# ==================================================================================================

reference_count_validation_rows = []


for reference_rule in REFERENCE_ORDER:
    observed_counts = (
        canonical_development_reference_table[
            reference_rule
        ]
        .value_counts()
        .sort_index()
        .reindex(
            CLASS_IDS,
            fill_value=0,
        )
        .to_dict()
    )

    expected_counts = (
        EXPECTED_DEVELOPMENT_REFERENCE_COUNTS[
            reference_rule
        ]
    )

    if observed_counts != expected_counts:
        raise RuntimeError(
            "Corrected development reference counts do not "
            "match the frozen Notebook 03 audit.\n"
            f"Reference: {reference_rule}\n"
            f"Expected: {expected_counts}\n"
            f"Observed: {observed_counts}"
        )

    for class_id in CLASS_IDS:
        reference_count_validation_rows.append(
            {
                "reference_rule": reference_rule,
                "class_id": class_id,
                "expected_count": int(
                    expected_counts[
                        class_id
                    ]
                ),
                "observed_count": int(
                    observed_counts[
                        class_id
                    ]
                ),
                "matches": True,
            }
        )


reference_count_validation_df = pd.DataFrame(
    reference_count_validation_rows
)


atomic_csv_save(
    reference_count_validation_df,
    MANIFEST_DIR
    / "frozen_notebook03_reference_count_validation.csv",
    index=False,
)


print("")
print("=" * 125)
print("CORRECTED DEVELOPMENT REFERENCE COUNTS")
print("=" * 125)

reference_count_display = (
    canonical_development_reference_table[
        REFERENCE_ORDER
    ]
    .apply(
        lambda column: (
            column.value_counts()
            .sort_index()
            .reindex(
                CLASS_IDS,
                fill_value=0,
            )
        )
    )
    .T
)

reference_count_display.columns = [
    "Mayo 0",
    "Mayo 1",
    "Mayo 2",
    "Mayo 3",
]

display(
    reference_count_display
)


# ==================================================================================================
# 18. Reference distributions
# ==================================================================================================

def build_reference_distribution_rows(
    patient_table,
    split,
):
    rows = []

    for reference_rule in REFERENCE_RULES:
        rule_key = reference_rule[
            "key"
        ]

        counts = (
            patient_table[
                rule_key
            ]
            .value_counts()
            .sort_index()
            .reindex(
                CLASS_IDS,
                fill_value=0,
            )
        )

        total = int(
            counts.sum()
        )

        for class_id in CLASS_IDS:
            rows.append(
                {
                    "split": split,
                    "reference_rule": (
                        rule_key
                    ),
                    "reference_display": (
                        reference_rule[
                            "display_name"
                        ]
                    ),
                    "reference_definition": (
                        reference_rule[
                            "definition"
                        ]
                    ),
                    "class_id": class_id,
                    "class_name": (
                        CLASS_NAMES[
                            class_id
                        ]
                    ),
                    "n_patients": int(
                        counts.loc[
                            class_id
                        ]
                    ),
                    "percent_patients": float(
                        100.0
                        *
                        counts.loc[
                            class_id
                        ]
                        /
                        total
                    ),
                }
            )

    return rows


development_reference_distribution_df = pd.DataFrame(
    build_reference_distribution_rows(
        canonical_development_reference_table,
        split="development_oof",
    )
)


official_reference_distribution_df = pd.DataFrame(
    build_reference_distribution_rows(
        canonical_official_reference_table,
        split="official_partition",
    )
)


atomic_csv_save(
    development_reference_distribution_df,
    V2_ROOT
    / "development_reference_distributions.csv",
    index=False,
)


atomic_csv_save(
    official_reference_distribution_df,
    V2_ROOT
    / "official_reference_distributions.csv",
    index=False,
)


# ==================================================================================================
# 19. Development crossed metrics
# ==================================================================================================

development_metric_rows = []
development_prediction_distribution_rows = []
development_confusion_rows = []


for (
    system,
    training_seed,
), patient_table in (
    development_patient_tables.items()
):
    for aggregation_rule in AGGREGATION_RULES:
        aggregation_key = aggregation_rule[
            "key"
        ]

        predicted_values = patient_table[
            aggregation_key
        ].to_numpy(
            dtype=int
        )

        predicted_counts = (
            pd.Series(
                predicted_values
            )
            .value_counts()
            .sort_index()
            .reindex(
                CLASS_IDS,
                fill_value=0,
            )
        )

        for class_id in CLASS_IDS:
            development_prediction_distribution_rows.append(
                {
                    "split": "development_oof",
                    "system": system,
                    "training_seed": training_seed,
                    "aggregation_rule": (
                        aggregation_key
                    ),
                    "aggregation_display": (
                        aggregation_rule[
                            "display_name"
                        ]
                    ),
                    "class_id": class_id,
                    "class_name": (
                        CLASS_NAMES[
                            class_id
                        ]
                    ),
                    "n_predicted": int(
                        predicted_counts.loc[
                            class_id
                        ]
                    ),
                    "percent_predicted": float(
                        100.0
                        *
                        predicted_counts.loc[
                            class_id
                        ]
                        /
                        EXPECTED_DEVELOPMENT_PATIENTS
                    ),
                }
            )

        for reference_rule in REFERENCE_RULES:
            reference_key = reference_rule[
                "key"
            ]

            reference_values = patient_table[
                reference_key
            ].to_numpy(
                dtype=int
            )

            metrics = compute_metrics(
                reference_values,
                predicted_values,
            )

            metric_row = {
                "split": "development_oof",
                "system": system,
                "training_seed": training_seed,
                "aggregation_rule": (
                    aggregation_key
                ),
                "aggregation_display": (
                    aggregation_rule[
                        "display_name"
                    ]
                ),
                "aggregation_origin": (
                    aggregation_rule[
                        "origin"
                    ]
                ),
                "reference_rule": (
                    reference_key
                ),
                "reference_display": (
                    reference_rule[
                        "display_name"
                    ]
                ),
                "reference_definition": (
                    reference_rule[
                        "definition"
                    ]
                ),
                "n_patients": int(
                    len(
                        patient_table
                    )
                ),
            }

            metric_row.update(
                metrics
            )

            development_metric_rows.append(
                metric_row
            )

            append_confusion_rows(
                output_rows=(
                    development_confusion_rows
                ),
                split="development_oof",
                system=system,
                training_seed=training_seed,
                aggregation_rule=aggregation_key,
                reference_rule=reference_key,
                y_true=reference_values,
                y_pred=predicted_values,
            )


development_metrics_by_seed_df = pd.DataFrame(
    development_metric_rows
)


development_prediction_distribution_df = (
    pd.DataFrame(
        development_prediction_distribution_rows
    )
)


development_confusion_df = pd.DataFrame(
    development_confusion_rows
)


atomic_csv_save(
    development_metrics_by_seed_df,
    V2_ROOT
    / "development_crossed_metrics_by_seed.csv",
    index=False,
)


atomic_csv_save(
    development_prediction_distribution_df,
    V2_ROOT
    / "development_prediction_distributions_by_seed.csv",
    index=False,
)


atomic_csv_save(
    development_confusion_df,
    V2_ROOT
    / "development_confusion_matrices_long.csv",
    index=False,
)


# ==================================================================================================
# 20. Development mean ± SD summary
# ==================================================================================================

development_summary_rows = []


development_group_columns = [
    "system",
    "aggregation_rule",
    "aggregation_display",
    "aggregation_origin",
    "reference_rule",
    "reference_display",
    "reference_definition",
]


for group_values, group_dataframe in (
    development_metrics_by_seed_df.groupby(
        development_group_columns,
        sort=False,
    )
):
    summary_row = {
        column_name: column_value
        for column_name, column_value in zip(
            development_group_columns,
            group_values,
        )
    }

    summary_row["n_seeds"] = int(
        group_dataframe[
            "training_seed"
        ].nunique()
    )

    summary_row["n_patients_per_seed"] = int(
        group_dataframe[
            "n_patients"
        ].iloc[
            0
        ]
    )

    for metric_name in METRIC_NAMES:
        values = group_dataframe[
            metric_name
        ].to_numpy(
            dtype=float
        )

        summary_row[
            metric_name
            +
            "_mean"
        ] = float(
            np.nanmean(
                values
            )
        )

        summary_row[
            metric_name
            +
            "_std"
        ] = float(
            np.nanstd(
                values,
                ddof=1,
            )
        )

        summary_row[
            metric_name
            +
            "_min"
        ] = float(
            np.nanmin(
                values
            )
        )

        summary_row[
            metric_name
            +
            "_max"
        ] = float(
            np.nanmax(
                values
            )
        )

    development_summary_rows.append(
        summary_row
    )


development_summary_df = pd.DataFrame(
    development_summary_rows
)


atomic_csv_save(
    development_summary_df,
    V2_ROOT
    / "development_crossed_metrics_summary.csv",
    index=False,
)


# ==================================================================================================
# 21. Official crossed point estimates
# ==================================================================================================

official_metric_rows = []
official_prediction_distribution_rows = []
official_confusion_rows = []


for system, patient_table in (
    official_patient_tables.items()
):
    for aggregation_rule in AGGREGATION_RULES:
        aggregation_key = aggregation_rule[
            "key"
        ]

        predicted_values = patient_table[
            aggregation_key
        ].to_numpy(
            dtype=int
        )

        predicted_counts = (
            pd.Series(
                predicted_values
            )
            .value_counts()
            .sort_index()
            .reindex(
                CLASS_IDS,
                fill_value=0,
            )
        )

        for class_id in CLASS_IDS:
            official_prediction_distribution_rows.append(
                {
                    "split": "official_partition",
                    "system": system,
                    "aggregation_rule": (
                        aggregation_key
                    ),
                    "aggregation_display": (
                        aggregation_rule[
                            "display_name"
                        ]
                    ),
                    "class_id": class_id,
                    "class_name": (
                        CLASS_NAMES[
                            class_id
                        ]
                    ),
                    "n_predicted": int(
                        predicted_counts.loc[
                            class_id
                        ]
                    ),
                    "percent_predicted": float(
                        100.0
                        *
                        predicted_counts.loc[
                            class_id
                        ]
                        /
                        EXPECTED_OFFICIAL_PATIENTS
                    ),
                }
            )

        for reference_rule in REFERENCE_RULES:
            reference_key = reference_rule[
                "key"
            ]

            reference_values = patient_table[
                reference_key
            ].to_numpy(
                dtype=int
            )

            metrics = compute_metrics(
                reference_values,
                predicted_values,
            )

            metric_row = {
                "split": "official_partition",
                "system": system,
                "aggregation_rule": (
                    aggregation_key
                ),
                "aggregation_display": (
                    aggregation_rule[
                        "display_name"
                    ]
                ),
                "aggregation_origin": (
                    aggregation_rule[
                        "origin"
                    ]
                ),
                "reference_rule": (
                    reference_key
                ),
                "reference_display": (
                    reference_rule[
                        "display_name"
                    ]
                ),
                "reference_definition": (
                    reference_rule[
                        "definition"
                    ]
                ),
                "n_patients": int(
                    len(
                        patient_table
                    )
                ),
            }

            metric_row.update(
                metrics
            )

            official_metric_rows.append(
                metric_row
            )

            append_confusion_rows(
                output_rows=(
                    official_confusion_rows
                ),
                split="official_partition",
                system=system,
                training_seed=np.nan,
                aggregation_rule=aggregation_key,
                reference_rule=reference_key,
                y_true=reference_values,
                y_pred=predicted_values,
            )


official_metrics_df = pd.DataFrame(
    official_metric_rows
)


official_prediction_distribution_df = (
    pd.DataFrame(
        official_prediction_distribution_rows
    )
)


official_confusion_df = pd.DataFrame(
    official_confusion_rows
)


atomic_csv_save(
    official_metrics_df,
    V2_ROOT
    / "official_crossed_metrics_point_estimates.csv",
    index=False,
)


atomic_csv_save(
    official_prediction_distribution_df,
    V2_ROOT
    / "official_prediction_distributions.csv",
    index=False,
)


atomic_csv_save(
    official_confusion_df,
    V2_ROOT
    / "official_confusion_matrices_long.csv",
    index=False,
)


# ==================================================================================================
# 22. Official reference-specific stratified bootstrap
#     Same RNG construction as version 1
# ==================================================================================================

def stratified_bootstrap_indices(
    y_true,
    rng,
):
    sampled_parts = []

    for class_id in CLASS_IDS:
        class_indices = np.flatnonzero(
            y_true
            ==
            class_id
        )

        if len(
            class_indices
        ) == 0:
            continue

        sampled_parts.append(
            rng.choice(
                class_indices,
                size=len(
                    class_indices
                ),
                replace=True,
            )
        )

    return np.concatenate(
        sampled_parts
    )


official_bootstrap_rows = []


print("")
print(
    "Running "
    +
    str(
        BOOTSTRAP_REPLICATES
    )
    +
    " reference-specific stratified patient-bootstrap replicates..."
)


for system_index, system in enumerate(
    [
        "Swin",
        "ViT",
    ]
):
    patient_table = official_patient_tables[
        system
    ]

    prediction_matrix = np.column_stack(
        [
            patient_table[
                aggregation_rule[
                    "key"
                ]
            ].to_numpy(
                dtype=int
            )
            for aggregation_rule in AGGREGATION_RULES
        ]
    )

    for reference_index, reference_rule in enumerate(
        REFERENCE_RULES
    ):
        reference_key = reference_rule[
            "key"
        ]

        y_true = patient_table[
            reference_key
        ].to_numpy(
            dtype=int
        )

        rng = np.random.default_rng(
            BOOTSTRAP_SEED
            +
            system_index
            *
            1000
            +
            reference_index
        )

        bootstrap_metric_arrays = {
            metric_name: np.full(
                (
                    BOOTSTRAP_REPLICATES,
                    len(
                        AGGREGATION_RULES
                    ),
                ),
                np.nan,
                dtype=float,
            )
            for metric_name in METRIC_NAMES
        }

        for bootstrap_index in range(
            BOOTSTRAP_REPLICATES
        ):
            sampled_indices = (
                stratified_bootstrap_indices(
                    y_true=y_true,
                    rng=rng,
                )
            )

            sampled_true = y_true[
                sampled_indices
            ]

            for aggregation_index in range(
                len(
                    AGGREGATION_RULES
                )
            ):
                sampled_predicted = (
                    prediction_matrix[
                        sampled_indices,
                        aggregation_index,
                    ]
                )

                matrix = confusion_matrix(
                    sampled_true,
                    sampled_predicted,
                    labels=CLASS_IDS,
                )

                metrics = (
                    metrics_from_confusion_matrix(
                        matrix
                    )
                )

                for metric_name in METRIC_NAMES:
                    bootstrap_metric_arrays[
                        metric_name
                    ][
                        bootstrap_index,
                        aggregation_index,
                    ] = metrics[
                        metric_name
                    ]

        for aggregation_index, aggregation_rule in enumerate(
            AGGREGATION_RULES
        ):
            point_row = official_metrics_df[
                (
                    official_metrics_df[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    official_metrics_df[
                        "aggregation_rule"
                    ]
                    ==
                    aggregation_rule[
                        "key"
                    ]
                )
                &
                (
                    official_metrics_df[
                        "reference_rule"
                    ]
                    ==
                    reference_key
                )
            ].iloc[
                0
            ]

            output_row = {
                "system": system,
                "aggregation_rule": (
                    aggregation_rule[
                        "key"
                    ]
                ),
                "aggregation_display": (
                    aggregation_rule[
                        "display_name"
                    ]
                ),
                "aggregation_origin": (
                    aggregation_rule[
                        "origin"
                    ]
                ),
                "reference_rule": (
                    reference_key
                ),
                "reference_display": (
                    reference_rule[
                        "display_name"
                    ]
                ),
                "reference_definition": (
                    reference_rule[
                        "definition"
                    ]
                ),
                "n_patients": int(
                    len(
                        patient_table
                    )
                ),
                "bootstrap_replicates": (
                    BOOTSTRAP_REPLICATES
                ),
                "bootstrap_stratification": (
                    reference_key
                ),
            }

            for metric_name in METRIC_NAMES:
                bootstrap_values = (
                    bootstrap_metric_arrays[
                        metric_name
                    ][
                        :,
                        aggregation_index,
                    ]
                )

                valid_values = bootstrap_values[
                    np.isfinite(
                        bootstrap_values
                    )
                ]

                output_row[
                    metric_name
                ] = float(
                    point_row[
                        metric_name
                    ]
                )

                output_row[
                    metric_name
                    +
                    "_ci_low"
                ] = float(
                    np.percentile(
                        valid_values,
                        2.5,
                    )
                )

                output_row[
                    metric_name
                    +
                    "_ci_high"
                ] = float(
                    np.percentile(
                        valid_values,
                        97.5,
                    )
                )

                output_row[
                    metric_name
                    +
                    "_n_valid_bootstrap"
                ] = int(
                    len(
                        valid_values
                    )
                )

            official_bootstrap_rows.append(
                output_row
            )

        print(
            "  completed "
            +
            system
            +
            " | "
            +
            reference_key
        )


official_metrics_ci_df = pd.DataFrame(
    official_bootstrap_rows
)


atomic_csv_save(
    official_metrics_ci_df,
    V2_ROOT
    / "official_crossed_metrics_with_bootstrap_ci.csv",
    index=False,
)


# ==================================================================================================
# 23. Metric ranges, primary-cell differences, and key cells
# ==================================================================================================

range_rows = []


for system in [
    "Swin",
    "ViT",
]:
    for (
        split_name,
        dataframe,
        metric_suffix,
    ) in [
        (
            "development_oof",
            development_summary_df,
            "_mean",
        ),
        (
            "official_partition",
            official_metrics_df,
            "",
        ),
    ]:
        system_dataframe = dataframe[
            dataframe[
                "system"
            ]
            ==
            system
        ]

        for metric_name in METRIC_NAMES:
            value_column = (
                metric_name
                +
                metric_suffix
            )

            values = system_dataframe[
                value_column
            ].to_numpy(
                dtype=float
            )

            primary_value = float(
                system_dataframe[
                    (
                        system_dataframe[
                            "aggregation_rule"
                        ]
                        ==
                        PRIMARY_AGGREGATION
                    )
                    &
                    (
                        system_dataframe[
                            "reference_rule"
                        ]
                        ==
                        PRIMARY_REFERENCE
                    )
                ][
                    value_column
                ].iloc[
                    0
                ]
            )

            minimum_row = system_dataframe.loc[
                system_dataframe[
                    value_column
                ].idxmin()
            ]

            maximum_row = system_dataframe.loc[
                system_dataframe[
                    value_column
                ].idxmax()
            ]

            range_rows.append(
                {
                    "split": split_name,
                    "system": system,
                    "metric": metric_name,
                    "primary_value": (
                        primary_value
                    ),
                    "minimum_value": float(
                        values.min()
                    ),
                    "minimum_aggregation": (
                        minimum_row[
                            "aggregation_rule"
                        ]
                    ),
                    "minimum_reference": (
                        minimum_row[
                            "reference_rule"
                        ]
                    ),
                    "maximum_value": float(
                        values.max()
                    ),
                    "maximum_aggregation": (
                        maximum_row[
                            "aggregation_rule"
                        ]
                    ),
                    "maximum_reference": (
                        maximum_row[
                            "reference_rule"
                        ]
                    ),
                    "full_crossed_range": float(
                        values.max()
                        -
                        values.min()
                    ),
                }
            )


range_summary_df = pd.DataFrame(
    range_rows
)


atomic_csv_save(
    range_summary_df,
    V2_ROOT
    / "crossed_metric_range_summary.csv",
    index=False,
)


key_combinations = [
    (
        "mean_probability_argmax",
        "modal_low",
        "locked_primary_cell",
    ),
    (
        "maximum_predicted_image_class",
        "modal_low",
        "maximum_prediction_vs_modal_reference",
    ),
    (
        "maximum_predicted_image_class",
        "maximum_label",
        "maximum_prediction_vs_maximum_reference",
    ),
    (
        "mean_probability_argmax",
        "maximum_label",
        "mean_prediction_vs_maximum_reference",
    ),
]


key_result_rows = []


for system in [
    "Swin",
    "ViT",
]:
    for (
        split_name,
        dataframe,
        qwk_column,
        mae_column,
    ) in [
        (
            "development_oof",
            development_summary_df,
            "qwk_mean",
            "mae_mean",
        ),
        (
            "official_partition",
            official_metrics_df,
            "qwk",
            "mae",
        ),
    ]:
        for (
            aggregation_rule,
            reference_rule,
            comparison_label,
        ) in key_combinations:
            row = dataframe[
                (
                    dataframe[
                        "system"
                    ]
                    ==
                    system
                )
                &
                (
                    dataframe[
                        "aggregation_rule"
                    ]
                    ==
                    aggregation_rule
                )
                &
                (
                    dataframe[
                        "reference_rule"
                    ]
                    ==
                    reference_rule
                )
            ].iloc[
                0
            ]

            key_result_rows.append(
                {
                    "split": split_name,
                    "system": system,
                    "comparison_label": (
                        comparison_label
                    ),
                    "aggregation_rule": (
                        aggregation_rule
                    ),
                    "reference_rule": (
                        reference_rule
                    ),
                    "qwk": float(
                        row[
                            qwk_column
                        ]
                    ),
                    "mae": float(
                        row[
                            mae_column
                        ]
                    ),
                }
            )


key_results_df = pd.DataFrame(
    key_result_rows
)


atomic_csv_save(
    key_results_df,
    V2_ROOT
    / "key_estimand_alignment_results.csv",
    index=False,
)


# ==================================================================================================
# 24. Between-system point differences
# ==================================================================================================

between_system_rows = []


for (
    split_name,
    dataframe,
    metric_suffix,
) in [
    (
        "development_oof",
        development_summary_df,
        "_mean",
    ),
    (
        "official_partition",
        official_metrics_df,
        "",
    ),
]:
    for aggregation_rule in AGGREGATION_ORDER:
        for reference_rule in REFERENCE_ORDER:
            swin_row = dataframe[
                (
                    dataframe[
                        "system"
                    ]
                    ==
                    "Swin"
                )
                &
                (
                    dataframe[
                        "aggregation_rule"
                    ]
                    ==
                    aggregation_rule
                )
                &
                (
                    dataframe[
                        "reference_rule"
                    ]
                    ==
                    reference_rule
                )
            ].iloc[
                0
            ]

            vit_row = dataframe[
                (
                    dataframe[
                        "system"
                    ]
                    ==
                    "ViT"
                )
                &
                (
                    dataframe[
                        "aggregation_rule"
                    ]
                    ==
                    aggregation_rule
                )
                &
                (
                    dataframe[
                        "reference_rule"
                    ]
                    ==
                    reference_rule
                )
            ].iloc[
                0
            ]

            output_row = {
                "split": split_name,
                "aggregation_rule": (
                    aggregation_rule
                ),
                "reference_rule": (
                    reference_rule
                ),
            }

            for metric_name in METRIC_NAMES:
                value_column = (
                    metric_name
                    +
                    metric_suffix
                )

                output_row[
                    metric_name
                    +
                    "_swin_minus_vit"
                ] = float(
                    swin_row[
                        value_column
                    ]
                    -
                    vit_row[
                        value_column
                    ]
                )

            between_system_rows.append(
                output_row
            )


between_system_differences_df = pd.DataFrame(
    between_system_rows
)


atomic_csv_save(
    between_system_differences_df,
    V2_ROOT
    / "between_system_differences_by_cell.csv",
    index=False,
)


# ==================================================================================================
# 25. Save matrix-form source data
# ==================================================================================================

def save_matrix_csvs():
    for system in [
        "Swin",
        "ViT",
    ]:
        development_system = (
            development_summary_df[
                development_summary_df[
                    "system"
                ]
                ==
                system
            ]
        )

        official_point_system = (
            official_metrics_df[
                official_metrics_df[
                    "system"
                ]
                ==
                system
            ]
        )

        official_ci_system = (
            official_metrics_ci_df[
                official_metrics_ci_df[
                    "system"
                ]
                ==
                system
            ]
        )

        for metric_name in METRIC_NAMES:
            for statistic in [
                "mean",
                "std",
            ]:
                value_column = (
                    metric_name
                    +
                    "_"
                    +
                    statistic
                )

                matrix = (
                    development_system.pivot(
                        index="aggregation_rule",
                        columns="reference_rule",
                        values=value_column,
                    )
                    .reindex(
                        index=AGGREGATION_ORDER,
                        columns=REFERENCE_ORDER,
                    )
                )

                atomic_csv_save(
                    matrix,
                    SOURCE_DATA_DIR
                    /
                    (
                        "development_"
                        +
                        system.lower()
                        +
                        "_"
                        +
                        metric_name
                        +
                        "_"
                        +
                        statistic
                        +
                        "_matrix.csv"
                    ),
                    index=True,
                )

            for suffix, source_dataframe, value_column in [
                (
                    "point",
                    official_point_system,
                    metric_name,
                ),
                (
                    "ci_low",
                    official_ci_system,
                    metric_name
                    +
                    "_ci_low",
                ),
                (
                    "ci_high",
                    official_ci_system,
                    metric_name
                    +
                    "_ci_high",
                ),
            ]:
                matrix = (
                    source_dataframe.pivot(
                        index="aggregation_rule",
                        columns="reference_rule",
                        values=value_column,
                    )
                    .reindex(
                        index=AGGREGATION_ORDER,
                        columns=REFERENCE_ORDER,
                    )
                )

                atomic_csv_save(
                    matrix,
                    SOURCE_DATA_DIR
                    /
                    (
                        "official_"
                        +
                        system.lower()
                        +
                        "_"
                        +
                        metric_name
                        +
                        "_"
                        +
                        suffix
                        +
                        "_matrix.csv"
                    ),
                    index=True,
                )


save_matrix_csvs()


# ==================================================================================================
# 26. Version 1 to version 2 audit
# ==================================================================================================

V1_DEVELOPMENT_SUMMARY = (
    V1_ROOT
    / "development_crossed_metrics_summary.csv"
)

V1_OFFICIAL_METRICS_CI = (
    V1_ROOT
    / "official_crossed_metrics_with_bootstrap_ci.csv"
)

V1_DEVELOPMENT_REFERENCE_DISTRIBUTIONS = (
    V1_ROOT
    / "development_reference_distributions.csv"
)

V1_OFFICIAL_REFERENCE_DISTRIBUTIONS = (
    V1_ROOT
    / "official_reference_distributions.csv"
)

V1_DEVELOPMENT_PREDICTION_DISTRIBUTIONS = (
    V1_ROOT
    / "development_prediction_distributions_by_seed.csv"
)

V1_OFFICIAL_PREDICTION_DISTRIBUTIONS = (
    V1_ROOT
    / "official_prediction_distributions.csv"
)


NONMEDIAN_REFERENCES = [
    "modal_low",
    "modal_high",
    "maximum_label",
]

CORRECTED_REFERENCES = [
    "median_low",
    "median_high",
]


def numeric_audit(
    old_dataframe,
    new_dataframe,
    keys,
    numeric_columns,
    audit_name,
):
    merged = old_dataframe.merge(
        new_dataframe,
        on=keys,
        how="outer",
        suffixes=(
            "_v1",
            "_v2",
        ),
        indicator=True,
    )

    if not (
        merged[
            "_merge"
        ]
        ==
        "both"
    ).all():
        raise RuntimeError(
            audit_name
            +
            " has unmatched version 1 or version 2 rows."
        )

    audit_rows = []

    for _, row in merged.iterrows():
        output_row = {
            key: row[
                key
            ]
            for key in keys
        }

        for numeric_column in numeric_columns:
            v1_value = float(
                row[
                    numeric_column
                    +
                    "_v1"
                ]
            )

            v2_value = float(
                row[
                    numeric_column
                    +
                    "_v2"
                ]
            )

            output_row[
                numeric_column
                +
                "_v1"
            ] = v1_value

            output_row[
                numeric_column
                +
                "_v2"
            ] = v2_value

            output_row[
                numeric_column
                +
                "_delta_v2_minus_v1"
            ] = (
                v2_value
                -
                v1_value
            )

        audit_rows.append(
            output_row
        )

    return pd.DataFrame(
        audit_rows
    )


if V1_DEVELOPMENT_SUMMARY.exists():
    v1_development_summary_df = pd.read_csv(
        V1_DEVELOPMENT_SUMMARY,
        low_memory=False,
    )

    development_audit_columns = []

    for metric_name in METRIC_NAMES:
        development_audit_columns.extend(
            [
                metric_name
                +
                "_mean",
                metric_name
                +
                "_std",
            ]
        )

    development_v1_v2_audit_df = numeric_audit(
        old_dataframe=(
            v1_development_summary_df
        ),
        new_dataframe=(
            development_summary_df
        ),
        keys=[
            "system",
            "aggregation_rule",
            "reference_rule",
        ],
        numeric_columns=(
            development_audit_columns
        ),
        audit_name=(
            "Development v1-v2 audit"
        ),
    )

    atomic_csv_save(
        development_v1_v2_audit_df,
        V2_ROOT
        / "v1_to_v2_development_metric_audit.csv",
        index=False,
    )

    unchanged_rows = (
        development_v1_v2_audit_df[
            development_v1_v2_audit_df[
                "reference_rule"
            ].isin(
                NONMEDIAN_REFERENCES
            )
        ]
    )

    delta_columns = [
        column
        for column in unchanged_rows.columns
        if column.endswith(
            "_delta_v2_minus_v1"
        )
    ]

    maximum_unexpected_change = float(
        unchanged_rows[
            delta_columns
        ].abs().to_numpy().max()
    )

    if maximum_unexpected_change > 1e-10:
        raise RuntimeError(
            "A nonmedian development result changed from v1 to v2.\n"
            "Only median-low and median-high rows are permitted to change.\n"
            f"Maximum unexpected absolute change: "
            f"{maximum_unexpected_change}"
        )

else:
    development_v1_v2_audit_df = pd.DataFrame()


if V1_OFFICIAL_METRICS_CI.exists():
    v1_official_metrics_ci_df = pd.read_csv(
        V1_OFFICIAL_METRICS_CI,
        low_memory=False,
    )

    official_audit_columns = []

    for metric_name in METRIC_NAMES:
        official_audit_columns.extend(
            [
                metric_name,
                metric_name
                +
                "_ci_low",
                metric_name
                +
                "_ci_high",
            ]
        )

    official_v1_v2_audit_df = numeric_audit(
        old_dataframe=(
            v1_official_metrics_ci_df
        ),
        new_dataframe=(
            official_metrics_ci_df
        ),
        keys=[
            "system",
            "aggregation_rule",
            "reference_rule",
        ],
        numeric_columns=(
            official_audit_columns
        ),
        audit_name=(
            "Official v1-v2 audit"
        ),
    )

    atomic_csv_save(
        official_v1_v2_audit_df,
        V2_ROOT
        / "v1_to_v2_official_metric_audit.csv",
        index=False,
    )

    unchanged_rows = (
        official_v1_v2_audit_df[
            official_v1_v2_audit_df[
                "reference_rule"
            ].isin(
                NONMEDIAN_REFERENCES
            )
        ]
    )

    delta_columns = [
        column
        for column in unchanged_rows.columns
        if column.endswith(
            "_delta_v2_minus_v1"
        )
    ]

    maximum_unexpected_change = float(
        unchanged_rows[
            delta_columns
        ].abs().to_numpy().max()
    )

    if maximum_unexpected_change > 1e-10:
        raise RuntimeError(
            "A nonmedian official result or CI changed from v1 to v2.\n"
            "Only median-low and median-high rows are permitted to change.\n"
            f"Maximum unexpected absolute change: "
            f"{maximum_unexpected_change}"
        )

else:
    official_v1_v2_audit_df = pd.DataFrame()


# Reference-distribution audit.
reference_distribution_audit_parts = []


for split_name, old_path, new_dataframe in [
    (
        "development_oof",
        V1_DEVELOPMENT_REFERENCE_DISTRIBUTIONS,
        development_reference_distribution_df,
    ),
    (
        "official_partition",
        V1_OFFICIAL_REFERENCE_DISTRIBUTIONS,
        official_reference_distribution_df,
    ),
]:
    if not old_path.exists():
        continue

    old_dataframe = pd.read_csv(
        old_path,
        low_memory=False,
    )

    merged = old_dataframe.merge(
        new_dataframe,
        on=[
            "reference_rule",
            "class_id",
        ],
        suffixes=(
            "_v1",
            "_v2",
        ),
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if not (
        merged[
            "_merge"
        ]
        ==
        "both"
    ).all():
        raise RuntimeError(
            "Reference-distribution rows do not align "
            "between v1 and v2."
        )

    merged[
        "split"
    ] = split_name

    merged[
        "count_delta_v2_minus_v1"
    ] = (
        merged[
            "n_patients_v2"
        ]
        -
        merged[
            "n_patients_v1"
        ]
    )

    reference_distribution_audit_parts.append(
        merged[
            [
                "split",
                "reference_rule",
                "class_id",
                "n_patients_v1",
                "n_patients_v2",
                "count_delta_v2_minus_v1",
            ]
        ]
    )


if reference_distribution_audit_parts:
    reference_distribution_audit_df = (
        pd.concat(
            reference_distribution_audit_parts,
            ignore_index=True,
        )
    )

    atomic_csv_save(
        reference_distribution_audit_df,
        V2_ROOT
        / "v1_to_v2_reference_distribution_audit.csv",
        index=False,
    )

    unexpected_reference_changes = (
        reference_distribution_audit_df[
            reference_distribution_audit_df[
                "reference_rule"
            ].isin(
                NONMEDIAN_REFERENCES
            )
            &
            (
                reference_distribution_audit_df[
                    "count_delta_v2_minus_v1"
                ]
                !=
                0
            )
        ]
    )

    if not unexpected_reference_changes.empty:
        raise RuntimeError(
            "A nonmedian reference distribution changed from v1 to v2."
        )

else:
    reference_distribution_audit_df = pd.DataFrame()


# Prediction distributions must be completely unchanged.
prediction_distribution_audit_parts = []


for split_name, old_path, new_dataframe, key_columns in [
    (
        "development_oof",
        V1_DEVELOPMENT_PREDICTION_DISTRIBUTIONS,
        development_prediction_distribution_df,
        [
            "system",
            "training_seed",
            "aggregation_rule",
            "class_id",
        ],
    ),
    (
        "official_partition",
        V1_OFFICIAL_PREDICTION_DISTRIBUTIONS,
        official_prediction_distribution_df,
        [
            "system",
            "aggregation_rule",
            "class_id",
        ],
    ),
]:
    if not old_path.exists():
        continue

    old_dataframe = pd.read_csv(
        old_path,
        low_memory=False,
    )

    merged = old_dataframe.merge(
        new_dataframe,
        on=key_columns,
        suffixes=(
            "_v1",
            "_v2",
        ),
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if not (
        merged[
            "_merge"
        ]
        ==
        "both"
    ).all():
        raise RuntimeError(
            "Prediction-distribution rows do not align "
            "between v1 and v2."
        )

    merged[
        "split"
    ] = split_name

    merged[
        "count_delta_v2_minus_v1"
    ] = (
        merged[
            "n_predicted_v2"
        ]
        -
        merged[
            "n_predicted_v1"
        ]
    )

    prediction_distribution_audit_parts.append(
        merged[
            [
                "split",
            ]
            +
            key_columns
            +
            [
                "n_predicted_v1",
                "n_predicted_v2",
                "count_delta_v2_minus_v1",
            ]
        ]
    )


if prediction_distribution_audit_parts:
    prediction_distribution_audit_df = (
        pd.concat(
            prediction_distribution_audit_parts,
            ignore_index=True,
        )
    )

    atomic_csv_save(
        prediction_distribution_audit_df,
        V2_ROOT
        / "v1_to_v2_prediction_distribution_audit.csv",
        index=False,
    )

    if (
        prediction_distribution_audit_df[
            "count_delta_v2_minus_v1"
        ]
        !=
        0
    ).any():
        raise RuntimeError(
            "An aggregation prediction distribution changed "
            "from v1 to v2. The correction should affect only "
            "reference labels."
        )

else:
    prediction_distribution_audit_df = pd.DataFrame()


# ==================================================================================================
# 27. Correction and guardrail records
# ==================================================================================================

v1_plan_path = (
    V1_ROOT
    / "analysis_plan_posthoc_crossed_sensitivity.json"
)


v1_plan_fingerprint = None


if v1_plan_path.exists():
    with open(
        v1_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        v1_plan_payload = json.load(
            file
        )

    v1_plan_fingerprint = (
        v1_plan_payload.get(
            "analysis_plan_fingerprint"
        )
    )


correction_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "superseded_version": (
        "11_aggregation_reference_crossed_sensitivity_v1"
    ),
    "authoritative_version": (
        "11_aggregation_reference_crossed_sensitivity_v2"
    ),
    "v1_plan_fingerprint": (
        v1_plan_fingerprint
    ),
    "v2_plan_fingerprint": (
        ANALYSIS_PLAN_FINGERPRINT
    ),
    "correction": {
        "v1_median_low": (
            "Lower middle observed image-level MES"
        ),
        "v1_median_high": (
            "Upper middle observed image-level MES"
        ),
        "v2_median_low": (
            "Floor of numerical median image-level MES"
        ),
        "v2_median_high": (
            "Ceiling of numerical median image-level MES"
        ),
        "reason": (
            "Version 2 restores consistency with the frozen "
            "Notebook 03 patient-reference audit."
        ),
    },
    "expected_affected_results": [
        "Rows or matrix columns using median_low",
        "Rows or matrix columns using median_high",
        "Median-reference distributions",
        "Median-reference confusion matrices",
        "Figures containing median-reference columns",
    ],
    "expected_unaffected_results": [
        "modal_low",
        "modal_high",
        "maximum_label",
        "All aggregation predictions",
        "Locked mean-probability/modal-low primary result",
        "Selected Swin and ViT systems",
        "Official-partition checkpoint ensemble",
    ],
    "primary_analysis_changed": False,
    "model_reselection_performed": False,
    "aggregation_reselection_performed": False,
    "reference_reselection_performed": False,
}


atomic_json_save(
    correction_record,
    V2_ROOT
    / "V1_TO_V2_CORRECTION_RECORD.json",
)


guardrail_payload = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "analysis_plan_fingerprint": (
        ANALYSIS_PLAN_FINGERPRINT
    ),
    "primary_locked_analysis": {
        "aggregation_rule": (
            PRIMARY_AGGREGATION
        ),
        "reference_rule": (
            PRIMARY_REFERENCE
        ),
        "changed": False,
    },
    "model_systems_changed": False,
    "checkpoint_membership_changed": False,
    "threshold_optimized": False,
    "calibration_estimated": False,
    "official_partition_used_for_selection": False,
    "interpretation": (
        "Version 2 is a definition correction restoring "
        "consistency with the frozen Notebook 03 median rules. "
        "All crossed official-partition results remain post-lock "
        "descriptive sensitivity estimates."
    ),
}


atomic_json_save(
    guardrail_payload,
    V2_ROOT
    / "NO_RESELECTION_GUARDRAIL.json",
)


supersession_text = """
NOTEBOOK 11 VERSION 2 SUPERSEDES VERSION 1
==========================================

Authoritative analysis:
11_aggregation_reference_crossed_sensitivity_v2

Superseded analysis:
11_aggregation_reference_crossed_sensitivity_v1

Reason:
Version 1 implemented median-low and median-high as the lower and
upper middle observed image labels. The frozen Notebook 03 audit
defined them as the floor and ceiling of the numerical median.
Version 2 restores those frozen definitions.

Affected outputs:
- median-low crossed cells
- median-high crossed cells
- median-reference distributions
- median-reference confusion matrices
- figures containing median columns

Unaffected outputs:
- modal-low
- modal-high
- maximum label
- all aggregation predictions
- selected Swin and ViT systems
- locked checkpoints
- locked mean-probability/modal-low primary analysis

Version 1 is retained for provenance only and must not be used to
reproduce the final manuscript.
""".strip()


atomic_text_save(
    supersession_text
    +
    "\n",
    V2_ROOT
    / "SUPERSEDES_VERSION_1.txt",
)


# ==================================================================================================
# 28. Publication figures
# ==================================================================================================

mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

sns.set_theme(
    style="white",
    context="paper",
)


PRIMARY_CELL_COLOR = "#E69F00"


def save_figure(
    figure,
    stem,
):
    if SAVE_PNG:
        figure.savefig(
            PNG_DIR
            /
            f"{stem}.png",
            dpi=PNG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

    if SAVE_SVG:
        figure.savefig(
            SVG_DIR
            /
            f"{stem}.svg",
            bbox_inches="tight",
            facecolor="white",
        )

    if SAVE_PDF:
        figure.savefig(
            PDF_DIR
            /
            f"{stem}.pdf",
            bbox_inches="tight",
            facecolor="white",
        )

    if SHOW_FIGURES:
        plt.show()

    plt.close(
        figure
    )


def draw_primary_cell_outline(
    axis,
):
    primary_row = AGGREGATION_ORDER.index(
        PRIMARY_AGGREGATION
    )

    primary_column = REFERENCE_ORDER.index(
        PRIMARY_REFERENCE
    )

    rectangle = patches.Rectangle(
        (
            primary_column,
            primary_row,
        ),
        1,
        1,
        fill=False,
        edgecolor=PRIMARY_CELL_COLOR,
        linewidth=3,
    )

    axis.add_patch(
        rectangle
    )


def plot_qwk_mae_heatmaps(
    dataframe,
    qwk_column,
    mae_column,
    title,
    stem,
):
    figure, axes = plt.subplots(
        2,
        2,
        figsize=(
            15,
            11,
        ),
    )

    for row_index, system in enumerate(
        [
            "Swin",
            "ViT",
        ]
    ):
        system_dataframe = dataframe[
            dataframe[
                "system"
            ]
            ==
            system
        ]

        qwk_matrix = (
            system_dataframe.pivot(
                index="aggregation_rule",
                columns="reference_rule",
                values=qwk_column,
            )
            .reindex(
                index=AGGREGATION_ORDER,
                columns=REFERENCE_ORDER,
            )
        )

        mae_matrix = (
            system_dataframe.pivot(
                index="aggregation_rule",
                columns="reference_rule",
                values=mae_column,
            )
            .reindex(
                index=AGGREGATION_ORDER,
                columns=REFERENCE_ORDER,
            )
        )

        qwk_axis = axes[
            row_index,
            0,
        ]

        mae_axis = axes[
            row_index,
            1,
        ]

        sns.heatmap(
            qwk_matrix,
            annot=True,
            fmt=".3f",
            cmap="viridis",
            linewidths=0.5,
            linecolor="white",
            xticklabels=[
                REFERENCE_LABELS[
                    key
                ]
                for key in REFERENCE_ORDER
            ],
            yticklabels=[
                AGGREGATION_LABELS[
                    key
                ]
                for key in AGGREGATION_ORDER
            ],
            cbar_kws={
                "label": "QWK",
            },
            ax=qwk_axis,
        )

        sns.heatmap(
            mae_matrix,
            annot=True,
            fmt=".3f",
            cmap="viridis_r",
            linewidths=0.5,
            linecolor="white",
            xticklabels=[
                REFERENCE_LABELS[
                    key
                ]
                for key in REFERENCE_ORDER
            ],
            yticklabels=[
                AGGREGATION_LABELS[
                    key
                ]
                for key in AGGREGATION_ORDER
            ],
            cbar_kws={
                "label": "MAE",
            },
            ax=mae_axis,
        )

        qwk_axis.set_title(
            system
            +
            ": QWK",
            fontweight="bold",
        )

        mae_axis.set_title(
            system
            +
            ": MAE",
            fontweight="bold",
        )

        for axis in [
            qwk_axis,
            mae_axis,
        ]:
            axis.set_xlabel(
                "Derived patient reference"
            )

            axis.set_ylabel(
                "Prediction aggregation"
            )

            axis.set_xticklabels(
                axis.get_xticklabels(),
                rotation=25,
                ha="right",
            )

            draw_primary_cell_outline(
                axis
            )

    figure.suptitle(
        title,
        fontsize=15,
        fontweight="bold",
        y=1.01,
    )

    figure.text(
        0.5,
        -0.01,
        (
            "Orange outline: locked mean-probability/modal-low "
            "primary combination. Rows and columns are displayed "
            "in a fixed conceptual order and are not ranked by performance."
        ),
        ha="center",
        va="top",
        fontsize=9.5,
    )

    figure.tight_layout()

    save_figure(
        figure,
        stem,
    )


plot_qwk_mae_heatmaps(
    dataframe=development_summary_df,
    qwk_column="qwk_mean",
    mae_column="mae_mean",
    title=(
        "Development OOF aggregation × reference sensitivity"
    ),
    stem=(
        "Figure_crossed_development_QWK_MAE_v2"
    ),
)


plot_qwk_mae_heatmaps(
    dataframe=official_metrics_df,
    qwk_column="qwk",
    mae_column="mae",
    title=(
        "Locked official-partition aggregation × "
        "reference sensitivity"
    ),
    stem=(
        "Figure_crossed_official_QWK_MAE_v2"
    ),
)


def plot_secondary_heatmaps(
    dataframe,
    metric_columns,
    title,
    stem,
):
    figure, axes = plt.subplots(
        2,
        3,
        figsize=(
            19,
            11,
        ),
    )

    for system_index, system in enumerate(
        [
            "Swin",
            "ViT",
        ]
    ):
        system_dataframe = dataframe[
            dataframe[
                "system"
            ]
            ==
            system
        ]

        for metric_index, (
            metric_title,
            metric_column,
        ) in enumerate(
            metric_columns
        ):
            matrix = (
                system_dataframe.pivot(
                    index="aggregation_rule",
                    columns="reference_rule",
                    values=metric_column,
                )
                .reindex(
                    index=AGGREGATION_ORDER,
                    columns=REFERENCE_ORDER,
                )
            )

            axis = axes[
                system_index,
                metric_index,
            ]

            sns.heatmap(
                matrix,
                annot=True,
                fmt=".3f",
                cmap="viridis",
                vmin=0,
                vmax=1,
                linewidths=0.5,
                linecolor="white",
                xticklabels=[
                    REFERENCE_LABELS[
                        key
                    ]
                    for key in REFERENCE_ORDER
                ],
                yticklabels=[
                    AGGREGATION_LABELS[
                        key
                    ]
                    for key in AGGREGATION_ORDER
                ],
                cbar_kws={
                    "label": metric_title,
                },
                ax=axis,
            )

            axis.set_title(
                system
                +
                ": "
                +
                metric_title,
                fontweight="bold",
            )

            axis.set_xlabel(
                "Derived patient reference"
            )

            axis.set_ylabel(
                "Prediction aggregation"
            )

            axis.set_xticklabels(
                axis.get_xticklabels(),
                rotation=25,
                ha="right",
            )

            draw_primary_cell_outline(
                axis
            )

    figure.suptitle(
        title,
        fontsize=15,
        fontweight="bold",
        y=1.01,
    )

    figure.tight_layout()

    save_figure(
        figure,
        stem,
    )


plot_secondary_heatmaps(
    dataframe=development_summary_df,
    metric_columns=[
        (
            "Accuracy",
            "accuracy_mean",
        ),
        (
            "Balanced accuracy",
            "balanced_accuracy_mean",
        ),
        (
            "Macro-F1",
            "macro_f1_mean",
        ),
    ],
    title=(
        "Development OOF crossed sensitivity: "
        "secondary patient metrics"
    ),
    stem=(
        "Figure_crossed_development_secondary_metrics_v2"
    ),
)


plot_secondary_heatmaps(
    dataframe=official_metrics_df,
    metric_columns=[
        (
            "Accuracy",
            "accuracy",
        ),
        (
            "Balanced accuracy",
            "balanced_accuracy",
        ),
        (
            "Macro-F1",
            "macro_f1",
        ),
    ],
    title=(
        "Locked official-partition crossed sensitivity: "
        "secondary patient metrics"
    ),
    stem=(
        "Figure_crossed_official_secondary_metrics_v2"
    ),
)


# ==================================================================================================
# 29. Publication-table helpers
# ==================================================================================================

def format_mean_sd(
    mean_value,
    sd_value,
):
    return (
        f"{float(mean_value):.4f} ± "
        f"{float(sd_value):.4f}"
    )


def format_estimate_ci(
    estimate,
    ci_low,
    ci_high,
):
    return (
        f"{float(estimate):.4f} "
        f"({float(ci_low):.4f}–"
        f"{float(ci_high):.4f})"
    )


def format_count_percent(
    count,
    percent,
):
    return (
        f"{int(count)} "
        f"({float(percent):.1f}%)"
    )


def format_count_mean_sd(
    mean_value,
    sd_value,
):
    return (
        f"{float(mean_value):.1f} ± "
        f"{float(sd_value):.1f}"
    )


def build_development_matrix(
    system,
    metric,
):
    subset = development_summary_df[
        development_summary_df[
            "system"
        ]
        ==
        system
    ].copy()

    subset[
        "formatted_value"
    ] = subset.apply(
        lambda row: format_mean_sd(
            row[
                metric
                +
                "_mean"
            ],
            row[
                metric
                +
                "_std"
            ],
        ),
        axis=1,
    )

    matrix = (
        subset.pivot(
            index="aggregation_rule",
            columns="reference_rule",
            values="formatted_value",
        )
        .reindex(
            index=AGGREGATION_ORDER,
            columns=REFERENCE_ORDER,
        )
    )

    matrix.index = [
        AGGREGATION_LABELS[
            key
        ]
        for key in matrix.index
    ]

    matrix.columns = [
        REFERENCE_LABELS[
            key
        ]
        for key in matrix.columns
    ]

    matrix.index.name = (
        "Aggregation rule"
    )

    return matrix.reset_index()


def build_official_long_table(
    system,
):
    subset = official_metrics_ci_df[
        official_metrics_ci_df[
            "system"
        ]
        ==
        system
    ].copy()

    aggregation_order_map = {
        value: index
        for index, value in enumerate(
            AGGREGATION_ORDER
        )
    }

    reference_order_map = {
        value: index
        for index, value in enumerate(
            REFERENCE_ORDER
        )
    }

    subset[
        "aggregation_order"
    ] = subset[
        "aggregation_rule"
    ].map(
        aggregation_order_map
    )

    subset[
        "reference_order"
    ] = subset[
        "reference_rule"
    ].map(
        reference_order_map
    )

    subset = subset.sort_values(
        [
            "aggregation_order",
            "reference_order",
        ]
    )

    rows = []

    for _, row in subset.iterrows():
        primary_estimand = (
            row[
                "aggregation_rule"
            ]
            ==
            PRIMARY_AGGREGATION
            and
            row[
                "reference_rule"
            ]
            ==
            PRIMARY_REFERENCE
        )

        if primary_estimand:
            if system == "Swin":
                status = (
                    "Locked primary system/estimand"
                )

            else:
                status = (
                    "Comparator under primary estimand"
                )

        else:
            status = (
                "Post-lock descriptive"
            )

        rows.append(
            {
                "Aggregation rule": (
                    AGGREGATION_LABELS[
                        row[
                            "aggregation_rule"
                        ]
                    ]
                ),
                "Derived reference": (
                    REFERENCE_LABELS[
                        row[
                            "reference_rule"
                        ]
                    ]
                ),
                "QWK (95% CI)": (
                    format_estimate_ci(
                        row[
                            "qwk"
                        ],
                        row[
                            "qwk_ci_low"
                        ],
                        row[
                            "qwk_ci_high"
                        ],
                    )
                ),
                "MAE (95% CI)": (
                    format_estimate_ci(
                        row[
                            "mae"
                        ],
                        row[
                            "mae_ci_low"
                        ],
                        row[
                            "mae_ci_high"
                        ],
                    )
                ),
                "Accuracy (95% CI)": (
                    format_estimate_ci(
                        row[
                            "accuracy"
                        ],
                        row[
                            "accuracy_ci_low"
                        ],
                        row[
                            "accuracy_ci_high"
                        ],
                    )
                ),
                "Balanced accuracy (95% CI)": (
                    format_estimate_ci(
                        row[
                            "balanced_accuracy"
                        ],
                        row[
                            "balanced_accuracy_ci_low"
                        ],
                        row[
                            "balanced_accuracy_ci_high"
                        ],
                    )
                ),
                "Macro-F1 (95% CI)": (
                    format_estimate_ci(
                        row[
                            "macro_f1"
                        ],
                        row[
                            "macro_f1_ci_low"
                        ],
                        row[
                            "macro_f1_ci_high"
                        ],
                    )
                ),
                "Status": status,
            }
        )

    return pd.DataFrame(
        rows
    )


def write_markdown_table(
    output_path,
    title,
    panels,
    note,
):
    sections = [
        "# "
        +
        title,
        "",
    ]

    for panel_title, dataframe in panels:
        sections.extend(
            [
                "## "
                +
                panel_title,
                "",
                dataframe.to_markdown(
                    index=False
                ),
                "",
            ]
        )

    sections.extend(
        [
            "**Note:** "
            +
            note,
            "",
        ]
    )

    atomic_text_save(
        "\n".join(
            sections
        ),
        output_path,
    )


# ==================================================================================================
# 30. Table S13
# ==================================================================================================

table_s13_panels = [
    (
        "Panel A. Swin QWK",
        build_development_matrix(
            "Swin",
            "qwk",
        ),
    ),
    (
        "Panel B. ViT QWK",
        build_development_matrix(
            "ViT",
            "qwk",
        ),
    ),
    (
        "Panel C. Swin MAE",
        build_development_matrix(
            "Swin",
            "mae",
        ),
    ),
    (
        "Panel D. ViT MAE",
        build_development_matrix(
            "ViT",
            "mae",
        ),
    ),
]


TABLE_S13_NOTE = (
    "Values are mean ± sample SD across training seeds 42, 2025, "
    "and 2026. Predictions from the five held-out folds were pooled "
    "within each seed before patient evaluation. Median-low and "
    "median-high denote the floor and ceiling, respectively, of the "
    "numerical median image-level MES. The mean-probability/modal-low "
    "cell is the prespecified primary estimand. Confidence-based rules "
    "were post hoc exploratory analyses. Higher QWK and lower MAE are "
    "better. Rows and columns are displayed in a fixed conceptual order."
)


write_markdown_table(
    PUBLICATION_TABLE_DIR
    / "Table_S13.md",
    "TABLE S13. DEVELOPMENT CROSSED QWK AND MAE",
    table_s13_panels,
    TABLE_S13_NOTE,
)


# ==================================================================================================
# 31. Table S14
# ==================================================================================================

table_s14_panels = []


for system in [
    "Swin",
    "ViT",
]:
    for metric_name, metric_title in [
        (
            "accuracy",
            "Exact accuracy",
        ),
        (
            "balanced_accuracy",
            "Balanced accuracy",
        ),
        (
            "macro_f1",
            "Macro-F1",
        ),
    ]:
        table_s14_panels.append(
            (
                system
                +
                " — "
                +
                metric_title,
                build_development_matrix(
                    system,
                    metric_name,
                ),
            )
        )


TABLE_S14_NOTE = (
    "Values are mean ± sample SD across three training seeds. "
    "Balanced accuracy is the unweighted mean of class-specific "
    "recall, and macro-F1 is the unweighted mean of class-specific "
    "F1. Median-low and median-high are the floor and ceiling of "
    "the numerical median image-level MES. Confidence-based rules "
    "were post hoc exploratory analyses."
)


write_markdown_table(
    PUBLICATION_TABLE_DIR
    / "Table_S14.md",
    (
        "TABLE S14. DEVELOPMENT CROSSED "
        "SECONDARY METRICS"
    ),
    table_s14_panels,
    TABLE_S14_NOTE,
)


# ==================================================================================================
# 32. Table S15
# ==================================================================================================

table_s15_swin = build_official_long_table(
    "Swin"
)

table_s15_vit = build_official_long_table(
    "ViT"
)


table_s15_panels = [
    (
        "Panel A. Swin",
        table_s15_swin,
    ),
    (
        "Panel B. ViT",
        table_s15_vit,
    ),
]


TABLE_S15_NOTE = (
    "Point estimates use the fixed locked official-partition "
    "probabilities. Confidence intervals are percentile intervals "
    "from 5,000 patient-bootstrap replicates stratified according "
    "to the derived reference defining each row. The mean-probability/"
    "modal-low combination is the prespecified primary estimand; "
    "Swin was the primary system and ViT the prespecified comparator. "
    "All other combinations are post-lock descriptive analyses. "
    "For the primary combination, authoritative predeclared intervals "
    "are reported in main Table V; intervals here were recomputed "
    "under the separately frozen crossed-analysis bootstrap and may "
    "differ slightly because of Monte Carlo resampling. QWK differences "
    "across references are not pure effect sizes because reference "
    "labels, marginal distributions, and expected agreement change."
)


write_markdown_table(
    PUBLICATION_TABLE_DIR
    / "Table_S15.md",
    (
        "TABLE S15. OFFICIAL-PARTITION "
        "CROSSED METRICS"
    ),
    table_s15_panels,
    TABLE_S15_NOTE,
)


# ==================================================================================================
# 33. Table S16
# ==================================================================================================

def build_reference_distribution_table(
    dataframe,
):
    working = dataframe.copy()

    working[
        "formatted"
    ] = working.apply(
        lambda row: format_count_percent(
            row[
                "n_patients"
            ],
            row[
                "percent_patients"
            ],
        ),
        axis=1,
    )

    matrix = (
        working.pivot(
            index="reference_rule",
            columns="class_id",
            values="formatted",
        )
        .reindex(
            index=REFERENCE_ORDER,
            columns=CLASS_IDS,
        )
    )

    matrix.index = [
        REFERENCE_LABELS[
            key
        ]
        for key in matrix.index
    ]

    matrix.columns = [
        CLASS_NAMES[
            class_id
        ]
        for class_id in matrix.columns
    ]

    matrix.index.name = (
        "Derived reference"
    )

    return matrix.reset_index()


development_reference_table = (
    build_reference_distribution_table(
        development_reference_distribution_df
    )
)


official_reference_table = (
    build_reference_distribution_table(
        official_reference_distribution_df
    )
)


development_prediction_summary = (
    development_prediction_distribution_df.groupby(
        [
            "system",
            "aggregation_rule",
            "class_id",
        ],
        as_index=False,
    )
    .agg(
        mean_count=(
            "n_predicted",
            "mean",
        ),
        sd_count=(
            "n_predicted",
            "std",
        ),
    )
)


development_prediction_summary[
    "formatted"
] = development_prediction_summary.apply(
    lambda row: format_count_mean_sd(
        row[
            "mean_count"
        ],
        row[
            "sd_count"
        ],
    ),
    axis=1,
)


development_prediction_matrix = (
    development_prediction_summary.pivot(
        index=[
            "system",
            "aggregation_rule",
        ],
        columns="class_id",
        values="formatted",
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                [
                    "Swin",
                    "ViT",
                ],
                AGGREGATION_ORDER,
            ],
            names=[
                "system",
                "aggregation_rule",
            ],
        )
    )
    .reset_index()
)


development_prediction_matrix[
    "aggregation_rule"
] = development_prediction_matrix[
    "aggregation_rule"
].map(
    AGGREGATION_LABELS
)


development_prediction_matrix = (
    development_prediction_matrix.rename(
        columns={
            "system": "System",
            "aggregation_rule": (
                "Aggregation rule"
            ),
            0: "Mayo 0",
            1: "Mayo 1",
            2: "Mayo 2",
            3: "Mayo 3",
        }
    )
)


official_prediction_matrix = (
    official_prediction_distribution_df.pivot(
        index=[
            "system",
            "aggregation_rule",
        ],
        columns="class_id",
        values="n_predicted",
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                [
                    "Swin",
                    "ViT",
                ],
                AGGREGATION_ORDER,
            ],
            names=[
                "system",
                "aggregation_rule",
            ],
        )
    )
    .reset_index()
)


official_prediction_matrix[
    "aggregation_rule"
] = official_prediction_matrix[
    "aggregation_rule"
].map(
    AGGREGATION_LABELS
)


official_prediction_matrix = (
    official_prediction_matrix.rename(
        columns={
            "system": "System",
            "aggregation_rule": (
                "Aggregation rule"
            ),
            0: "Mayo 0",
            1: "Mayo 1",
            2: "Mayo 2",
            3: "Mayo 3",
        }
    )
)


for class_column in [
    "Mayo 0",
    "Mayo 1",
    "Mayo 2",
    "Mayo 3",
]:
    official_prediction_matrix[
        class_column
    ] = official_prediction_matrix[
        class_column
    ].astype(int)


table_s16_panels = [
    (
        "Panel A. Development reference distributions",
        development_reference_table,
    ),
    (
        "Panel B. Official-partition reference distributions",
        official_reference_table,
    ),
    (
        (
            "Panel C. Development prediction distributions "
            "(mean ± SD across seeds)"
        ),
        development_prediction_matrix,
    ),
    (
        (
            "Panel D. Official-partition prediction distributions"
        ),
        official_prediction_matrix,
    ),
]


TABLE_S16_NOTE = (
    "Reference-distribution cells show patient count and percentage. "
    "Median-low and median-high denote the floor and ceiling of the "
    "numerical median image-level MES. Development prediction "
    "distributions show mean ± sample SD of patient counts across "
    "three seeds, each containing 478 patients. Official-partition "
    "prediction distributions show exact counts among 86 patients. "
    "Prediction distributions depend on aggregation but not on the "
    "derived reference."
)


write_markdown_table(
    PUBLICATION_TABLE_DIR
    / "Table_S16.md",
    (
        "TABLE S16. REFERENCE AND "
        "PREDICTION DISTRIBUTIONS"
    ),
    table_s16_panels,
    TABLE_S16_NOTE,
)


# ==================================================================================================
# 34. Table S17
# ==================================================================================================

SELECTED_CONFUSION_COMBINATIONS = [
    (
        "mean_probability_argmax",
        "modal_low",
        "Mean probability × modal-low",
    ),
    (
        "maximum_predicted_image_class",
        "modal_low",
        "Maximum prediction × modal-low",
    ),
    (
        "mean_probability_argmax",
        "maximum_label",
        "Mean probability × maximum label",
    ),
    (
        "maximum_predicted_image_class",
        "maximum_label",
        "Maximum prediction × maximum label",
    ),
]


def label_confusion_matrix(
    matrix,
):
    output = matrix.copy()

    output.index = [
        CLASS_NAMES[
            int(
                value
            )
        ]
        for value in output.index
    ]

    output.columns = [
        "Pred. "
        +
        CLASS_NAMES[
            int(
                value
            )
        ]
        for value in output.columns
    ]

    output.index.name = (
        "Reference class"
    )

    return output.reset_index()


def build_development_confusion_table(
    system,
    aggregation_rule,
    reference_rule,
):
    subset = development_confusion_df[
        (
            development_confusion_df[
                "system"
            ]
            ==
            system
        )
        &
        (
            development_confusion_df[
                "aggregation_rule"
            ]
            ==
            aggregation_rule
        )
        &
        (
            development_confusion_df[
                "reference_rule"
            ]
            ==
            reference_rule
        )
    ].copy()

    summary = (
        subset.groupby(
            [
                "true_class",
                "predicted_class",
            ],
            as_index=False,
        )
        .agg(
            mean_count=(
                "count",
                "mean",
            ),
            sd_count=(
                "count",
                "std",
            ),
        )
    )

    summary[
        "formatted"
    ] = summary.apply(
        lambda row: format_count_mean_sd(
            row[
                "mean_count"
            ],
            row[
                "sd_count"
            ],
        ),
        axis=1,
    )

    matrix = (
        summary.pivot(
            index="true_class",
            columns="predicted_class",
            values="formatted",
        )
        .reindex(
            index=CLASS_IDS,
            columns=CLASS_IDS,
        )
    )

    return label_confusion_matrix(
        matrix
    )


def build_official_confusion_table(
    system,
    aggregation_rule,
    reference_rule,
):
    subset = official_confusion_df[
        (
            official_confusion_df[
                "system"
            ]
            ==
            system
        )
        &
        (
            official_confusion_df[
                "aggregation_rule"
            ]
            ==
            aggregation_rule
        )
        &
        (
            official_confusion_df[
                "reference_rule"
            ]
            ==
            reference_rule
        )
    ].copy()

    matrix = (
        subset.pivot(
            index="true_class",
            columns="predicted_class",
            values="count",
        )
        .reindex(
            index=CLASS_IDS,
            columns=CLASS_IDS,
            fill_value=0,
        )
        .astype(int)
    )

    return label_confusion_matrix(
        matrix
    )


table_s17_panels = []


for split_name in [
    "Development OOF",
    "Official partition",
]:
    for system in [
        "Swin",
        "ViT",
    ]:
        for (
            aggregation_rule,
            reference_rule,
            combination_label,
        ) in SELECTED_CONFUSION_COMBINATIONS:
            if split_name == "Development OOF":
                table = (
                    build_development_confusion_table(
                        system,
                        aggregation_rule,
                        reference_rule,
                    )
                )

            else:
                table = (
                    build_official_confusion_table(
                        system,
                        aggregation_rule,
                        reference_rule,
                    )
                )

            table_s17_panels.append(
                (
                    split_name
                    +
                    " — "
                    +
                    system
                    +
                    " — "
                    +
                    combination_label,
                    table,
                )
            )


TABLE_S17_NOTE = (
    "Development cells show mean ± sample SD of patient counts "
    "across three seeds; displayed row totals may differ from integer "
    "reference support by 0.1 because cell means are rounded to one "
    "decimal place. Official-partition cells show exact counts. "
    "The matrices represent two aligned and two deliberately "
    "mismatched mean/maximum aggregation-reference combinations. "
    "Complete matrices are provided in the machine-readable long-form files."
)


write_markdown_table(
    PUBLICATION_TABLE_DIR
    / "Table_S17.md",
    (
        "TABLE S17. SELECTED CROSSED "
        "CONFUSION MATRICES"
    ),
    table_s17_panels,
    TABLE_S17_NOTE,
)


# Preserve complete confusion matrices as supplementary data.
shutil.copy2(
    V2_ROOT
    / "development_confusion_matrices_long.csv",
    PUBLICATION_TABLE_DIR
    / (
        "Supplementary_Data_S17_"
        "development_all_confusion_matrices_v2.csv"
    ),
)


shutil.copy2(
    V2_ROOT
    / "official_confusion_matrices_long.csv",
    PUBLICATION_TABLE_DIR
    / (
        "Supplementary_Data_S17_"
        "official_all_confusion_matrices_v2.csv"
    ),
)


# ==================================================================================================
# 35. Excel workbook for Tables S13–S17
# ==================================================================================================

workbook_path = (
    PUBLICATION_TABLE_DIR
    / "Tables_S13_to_S17_v2.xlsx"
)


with pd.ExcelWriter(
    workbook_path,
    engine="openpyxl",
) as writer:
    readme_rows = [
        {
            "Table": "S13",
            "Title": (
                "Development crossed QWK and MAE"
            ),
            "Source": (
                "development_crossed_metrics_summary.csv"
            ),
        },
        {
            "Table": "S14",
            "Title": (
                "Development crossed secondary metrics"
            ),
            "Source": (
                "development_crossed_metrics_summary.csv"
            ),
        },
        {
            "Table": "S15",
            "Title": (
                "Official-partition crossed metrics"
            ),
            "Source": (
                "official_crossed_metrics_with_bootstrap_ci.csv"
            ),
        },
        {
            "Table": "S16",
            "Title": (
                "Reference and prediction distributions"
            ),
            "Source": (
                "Reference and prediction-distribution CSV files"
            ),
        },
        {
            "Table": "S17",
            "Title": (
                "Selected crossed confusion matrices"
            ),
            "Source": (
                "Development and official long-form confusion CSV files"
            ),
        },
    ]

    pd.DataFrame(
        readme_rows
    ).to_excel(
        writer,
        sheet_name="README",
        index=False,
    )

    s13_sheet_names = [
        "S13A_Swin_QWK",
        "S13B_ViT_QWK",
        "S13C_Swin_MAE",
        "S13D_ViT_MAE",
    ]

    for sheet_name, (
        _,
        dataframe,
    ) in zip(
        s13_sheet_names,
        table_s13_panels,
    ):
        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )

    s14_sheet_names = [
        "S14A_Swin_Accuracy",
        "S14B_Swin_BalAcc",
        "S14C_Swin_MacroF1",
        "S14D_ViT_Accuracy",
        "S14E_ViT_BalAcc",
        "S14F_ViT_MacroF1",
    ]

    for sheet_name, (
        _,
        dataframe,
    ) in zip(
        s14_sheet_names,
        table_s14_panels,
    ):
        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )

    table_s15_swin.to_excel(
        writer,
        sheet_name="S15A_Swin",
        index=False,
    )

    table_s15_vit.to_excel(
        writer,
        sheet_name="S15B_ViT",
        index=False,
    )

    development_reference_table.to_excel(
        writer,
        sheet_name="S16A_Dev_References",
        index=False,
    )

    official_reference_table.to_excel(
        writer,
        sheet_name="S16B_Off_References",
        index=False,
    )

    development_prediction_matrix.to_excel(
        writer,
        sheet_name="S16C_Dev_Predictions",
        index=False,
    )

    official_prediction_matrix.to_excel(
        writer,
        sheet_name="S16D_Off_Predictions",
        index=False,
    )

    for panel_index, (
        panel_title,
        dataframe,
    ) in enumerate(
        table_s17_panels,
        start=1,
    ):
        sheet_name = (
            "S17_"
            +
            f"{panel_index:02d}"
        )

        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
            startrow=2,
        )

        worksheet = writer.book[
            sheet_name
        ]

        worksheet[
            "A1"
        ] = panel_title


# Basic workbook formatting.
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook


workbook = load_workbook(
    workbook_path
)


header_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)


for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )

        cell.fill = header_fill

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for column_index, column_cells in enumerate(
        worksheet.columns,
        start=1,
    ):
        maximum_length = 0

        for cell in column_cells:
            if cell.value is None:
                continue

            maximum_length = max(
                maximum_length,
                len(
                    str(
                        cell.value
                    )
                ),
            )

        worksheet.column_dimensions[
            get_column_letter(
                column_index
            )
        ].width = min(
            max(
                maximum_length
                +
                2,
                12,
            ),
            45,
        )


workbook.save(
    workbook_path
)


# ==================================================================================================
# 36. Combined text export
# ==================================================================================================

markdown_paths = [
    PUBLICATION_TABLE_DIR
    / "Table_S13.md",
    PUBLICATION_TABLE_DIR
    / "Table_S14.md",
    PUBLICATION_TABLE_DIR
    / "Table_S15.md",
    PUBLICATION_TABLE_DIR
    / "Table_S16.md",
    PUBLICATION_TABLE_DIR
    / "Table_S17.md",
]


combined_sections = []


for markdown_path in markdown_paths:
    combined_sections.append(
        "\n"
        +
        "="
        *
        120
        +
        "\nSOURCE FILE: "
        +
        markdown_path.name
        +
        "\n"
        +
        "="
        *
        120
        +
        "\n\n"
        +
        markdown_path.read_text(
            encoding="utf-8"
        )
    )


combined_table_text = "\n".join(
    combined_sections
)


atomic_text_save(
    combined_table_text,
    PUBLICATION_TABLE_DIR
    / "Tables_S13_to_S17_v2_combined.txt",
)


# ==================================================================================================
# 37. Final generated-output manifest
# ==================================================================================================

generated_files = sorted(
    [
        path
        for path in V2_ROOT.rglob(
            "*"
        )
        if (
            path.is_file()
            and
            not path.name.endswith(
                ".temporary"
            )
        )
    ]
)


generated_manifest_rows = []


for path in generated_files:
    generated_manifest_rows.append(
        {
            "relative_path": str(
                path.relative_to(
                    V2_ROOT
                )
            ),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": file_sha256(
                path
            ),
        }
    )


generated_manifest_df = pd.DataFrame(
    generated_manifest_rows
)


atomic_csv_save(
    generated_manifest_df,
    MANIFEST_DIR
    / "generated_output_manifest.csv",
    index=False,
)


run_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "11_aggregation_reference_crossed_sensitivity_v2.ipynb"
    ),
    "analysis_plan_fingerprint": (
        ANALYSIS_PLAN_FINGERPRINT
    ),
    "supersedes": (
        "11_aggregation_reference_crossed_sensitivity_v1"
    ),
    "input_source_manifest": str(
        V1_SOURCE_MANIFEST
    ),
    "output_root": str(
        V2_ROOT
    ),
    "systems": [
        "Swin",
        "ViT",
    ],
    "training_seeds": TRAINING_SEEDS,
    "development_images_per_seed": (
        EXPECTED_DEVELOPMENT_IMAGES
    ),
    "development_patients_per_seed": (
        EXPECTED_DEVELOPMENT_PATIENTS
    ),
    "official_images": (
        EXPECTED_OFFICIAL_IMAGES
    ),
    "official_patients": (
        EXPECTED_OFFICIAL_PATIENTS
    ),
    "bootstrap_replicates": (
        BOOTSTRAP_REPLICATES
    ),
    "bootstrap_seed": (
        BOOTSTRAP_SEED
    ),
    "median_definitions": {
        "median_low": (
            "floor of numerical median image-level MES"
        ),
        "median_high": (
            "ceiling of numerical median image-level MES"
        ),
    },
    "safety": {
        "training_performed": False,
        "checkpoints_loaded": False,
        "raw_images_accessed": False,
        "model_inference_performed": False,
        "calibration_fitted": False,
        "threshold_optimized": False,
        "model_selection_performed": False,
        "aggregation_reselection_performed": False,
        "reference_reselection_performed": False,
        "primary_analysis_changed": False,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": mpl.__version__,
        "seaborn": sns.__version__,
    },
}


atomic_json_save(
    run_manifest,
    MANIFEST_DIR
    / "crossed_sensitivity_v2_run_manifest.json",
)


# ==================================================================================================
# 38. Final result display
# ==================================================================================================

print("")
print("=" * 125)
print("KEY ESTIMAND-ALIGNMENT RESULTS — VERSION 2")
print("=" * 125)

display(
    key_results_df.sort_values(
        [
            "split",
            "system",
            "comparison_label",
        ]
    )
)


print("")
print("=" * 125)
print("CROSSED METRIC RANGES — VERSION 2")
print("=" * 125)

display(
    range_summary_df.sort_values(
        [
            "split",
            "system",
            "metric",
        ]
    )
)


print("")
print("=" * 125)
print("CORRECTED DEVELOPMENT REFERENCE DISTRIBUTIONS")
print("=" * 125)

display(
    development_reference_table
)


# ==================================================================================================
# 39. Syntax-safe completion summary
# ==================================================================================================

summary_lines = [
    "11_aggregation_reference_crossed_sensitivity_v2 COMPLETED",
    "=========================================================",
    "",
    "Correction:",
    (
        "Median-low is floor of numerical median; "
        "median-high is ceiling of numerical median."
    ),
    "",
    "Supersedes:",
    str(
        V1_ROOT
    ),
    "",
    "Version 2 output:",
    str(
        V2_ROOT
    ),
    "",
    "Analysis-plan fingerprint:",
    ANALYSIS_PLAN_FINGERPRINT,
    "",
    "Development median-low counts:",
    str(
        EXPECTED_DEVELOPMENT_REFERENCE_COUNTS[
            "median_low"
        ]
    ),
    "",
    "Development median-high counts:",
    str(
        EXPECTED_DEVELOPMENT_REFERENCE_COUNTS[
            "median_high"
        ]
    ),
    "",
    "Training performed:",
    "NO",
    "",
    "Checkpoint loading performed:",
    "NO",
    "",
    "Raw-image access performed:",
    "NO",
    "",
    "Model inference performed:",
    "NO",
    "",
    "Calibration fitting performed:",
    "NO",
    "",
    "Threshold optimization performed:",
    "NO",
    "",
    "Model reselection performed:",
    "NO",
    "",
    "Aggregation reselection performed:",
    "NO",
    "",
    "Reference reselection performed:",
    "NO",
    "",
    "Primary analysis changed:",
    "NO",
    "",
    "Development crossed summary:",
    str(
        V2_ROOT
        / "development_crossed_metrics_summary.csv"
    ),
    "",
    "Official crossed metrics with CIs:",
    str(
        V2_ROOT
        / "official_crossed_metrics_with_bootstrap_ci.csv"
    ),
    "",
    "Version 1 to version 2 audit:",
    str(
        V2_ROOT
        / "v1_to_v2_development_metric_audit.csv"
    ),
    str(
        V2_ROOT
        / "v1_to_v2_official_metric_audit.csv"
    ),
    "",
    "Publication tables:",
    str(
        workbook_path
    ),
    "",
    "Combined text export:",
    str(
        PUBLICATION_TABLE_DIR
        / "Tables_S13_to_S17_v2_combined.txt"
    ),
    "",
    "Figures:",
    str(
        FIGURE_ROOT
    ),
]


summary_text = "\n".join(
    summary_lines
)


atomic_text_save(
    summary_text
    +
    "\n",
    V2_ROOT
    / "CROSSED_SENSITIVITY_V2_SUMMARY.txt",
)


print("")
print("=" * 125)
print(
    "11_aggregation_reference_crossed_sensitivity_v2 "
    "COMPLETED SUCCESSFULLY"
)
print("=" * 125)

print("")
print("Version 2 output directory:")
print(V2_ROOT)

print("")
print("Analysis-plan fingerprint:")
print(ANALYSIS_PLAN_FINGERPRINT)

print("")
print("Corrected publication workbook:")
print(workbook_path)

print("")
print("Corrected combined Markdown/text export:")
print(
    PUBLICATION_TABLE_DIR
    / "Tables_S13_to_S17_v2_combined.txt"
)

print("")
print("Correction record:")
print(
    V2_ROOT
    / "V1_TO_V2_CORRECTION_RECORD.json"
)

print("")
print("Supersession notice:")
print(
    V2_ROOT
    / "SUPERSEDES_VERSION_1.txt"
)

print("")
print("Safety confirmation:")
print(
    "No training, checkpoint loading, raw-image access, "
    "model inference, calibration fitting, threshold optimization, "
    "model selection, aggregation reselection, or reference "
    "reselection occurred."
)